# LATE FUSION + ENSEMBLE VALIDATION/TEST - compatibile con training

Questo notebook usa checkpoint già allenati dal notebook `UPSAMPLING_SUBSAMPLING.ipynb`.

Non allena le reti base. Ricostruisce gli stessi split del notebook di training, calcola probabilità finali ed embedding dei modelli base, valida più combinazioni e poi testa le combinazioni selezionate.

Flussi inclusi:

1. **Ensemble sulle probabilità finali**: soft voting uniforme, weighted soft voting globale, weighted soft voting per classe.
2. **Late fusion sugli embedding**: concatenazione delle feature prima della classification head + classificatore finale leggero.

## Compatibilità con subsampling

Il notebook è adattabile anche al protocollo `subsampling` in modo più rigoroso rispetto alla versione precedente:

- per la **validation delle combinazioni** usa checkpoint fold-specific, per esempio `model_fold_0.pt`, coerenti con `VAL_FOLD_FOR_SUBSAMPLING`;
- per il **test finale** usa checkpoint `final_refit`, per esempio `model_final_refit.pt`, valutati sul test set bilanciato separato;
- per la late fusion finale nel subsampling, il classificatore finale viene ri-addestrato sugli embedding del `cv_pool` usando i checkpoint `final_refit`, poi viene testato sul test set separato.

In questo modo il notebook resta compatibile con gli output: non richiede cartelle `generated_metadata` e non reintroduce training delle reti base.


## Nuova modalità: screening da probabilità già salvate

Se hai già i CSV `probabilities_*.csv` prodotti dal notebook di training, puoi usarli direttamente per creare una tabella riepilogativa dei modelli e delle combinazioni più promettenti sul validation set. In questa modalità non vengono ricaricati checkpoint, non vengono rilette immagini e non viene allenata nessuna rete base: il notebook lavora solo sulle probabilità di appartenenza alle classi già salvate.

Output condensati:
- `validation_model_screening_single_models.csv`
- `validation_probability_ensemble_screening_ranked.csv`
- `validation_probability_ensemble_screening_summary.csv`
- `validation_probability_screening_summary.json`


## Migliorie v6

Questa versione aggiunge una modalità di screening validation più pulita:

- non stampa più DataFrame enormi a fine run;
- assegna `very_promising`/`promising` confrontando gli ensemble anche con il miglior modello singolo globale, non solo con i migliori modelli dentro la stessa combinazione;
- salva una tabella `validation_probability_selected_for_test.csv` con poche combinazioni candidate da testare;
- mantiene separati i flussi da file di probabilità e quelli da checkpoint/late fusion.


## Nota sulle metriche di validation screening

Le tabelle prodotte dalla modalità `RUN_VALIDATION_PROBABILITY_SCREENING_FROM_FILES` sono tabelle di **selezione su validation**, non tabelle di test finale.

- Con `SAMPLING_MODE = "upsampling"`, le metriche sono calcolate sul validation set hold-out non augmentato.
- Con `SAMPLING_MODE = "subsampling"`, le metriche sono calcolate sulle probabilità **OOF** della cross-validation; quindi vanno confrontate con la media CV/OOF dei singoli modelli, non con la test accuracy del modello `final_refit`.

Per questo nel CSV vengono aggiunte colonne esplicite come `validation_screening_accuracy` e `delta_validation_screening_accuracy_pp_vs_best_global_single`.


In [ ]:
# ============================================================
# 1. CONFIGURAZIONE ENSEMBLE + LATE FUSION
# ============================================================
# Questo notebook NON allena ResNet / ViT / DINOv2.
# I checkpoint devono essere già stati prodotti dal notebook di training.
#
# Compatibilità con training:
# - nessuna opzione background blur;
# - nessuna cartella results/.../generated_metadata richiesta;
# - gli split vengono ricostruiti con stessa logica, seed e manifest upsampling.

from pathlib import Path

REPO_ROOT = Path.cwd().resolve()

CONFIG = {
    # "upsampling" oppure "subsampling".
    # Per replicare il risultato principale della relazione usa normalmente "upsampling".
    "SAMPLING_MODE": "upsampling",

    # Dataset originale usato dal notebook training.
    "DATASET_DRIVE_DIR": str(REPO_ROOT / "_external_dataset"),
    "DATASET_LOCAL_DIR": str(REPO_ROOT / "_external_dataset"),

    # Se True, prova a usare copie locali/tar in /content per velocizzare la lettura immagini.
    # Se i file locali non esistono, il notebook mantiene automaticamente i path su Drive.
    "USE_TAR_LOCAL_DATASET": True,
    "TAR_ORIGINAL_PATH": str(REPO_ROOT / "_external_dataset" / "Comics_original_224.tar"),

    # Opzionale: tar della cartella augmented_train_only_original_seed42_test0p1.
    # Se non esiste, gli augmented vengono letti direttamente da Drive.
    "USE_TAR_AUGMENTED_UPSAMPLING": False,
    "TAR_AUG_ORIGINAL_PATH": str(REPO_ROOT / "_external_dataset" / "augmented_train_only_original_seed42_test0p1.tar"),

    # Devono coincidere con il notebook di training.
    "SEED": 42,
    "TEST_SIZE": 0.10,
    "N_SPLITS": 5,

    # Per subsampling: fold usato come validation interna.
    # Il notebook userà i checkpoint <model>_fold_<VAL_FOLD>.pt per validare le combinazioni
    # e i checkpoint <model>_final_refit.pt per il test finale separato.
    "VAL_FOLD_FOR_SUBSAMPLING": 0,
    "SUBSAMPLING_USE_FOLD_CHECKPOINTS_FOR_VALIDATION": True,
    "SUBSAMPLING_USE_FINAL_REFIT_FOR_TEST": True,

    # Nel subsampling, per testare la late fusion in modo più sensato, il classificatore
    # finale viene ri-addestrato sugli embedding del cv_pool prodotti dai checkpoint final_refit.
    # Con MLP si usa uno split interno train/val solo per early stopping del classificatore finale.
    "REFIT_FUSION_CLASSIFIER_FOR_SUBSAMPLING_TEST": True,
    "FINAL_FUSION_VAL_SIZE": 0.1111,

    # Solo informativo/controllo di qualità: con checkpoint final_refit la validation
    # subsampling non è rigorosa per selezionare combinazioni, perché i modelli base
    # hanno già visto il CV pool. Il test set rimane separato.
    "SUBSAMPLING_CHECKPOINT_USAGE": "final_refit",

    # Output del notebook.
    "OUTPUT_DIR": str(REPO_ROOT / "results" / "ensemble_late_fusion"),
    "FORCE_REEXTRACT_OUTPUTS": False,

    # ========================================================
    # SCREENING VALIDATION DA PROBABILITÀ GIÀ SALVATE
    # ========================================================
    # Usa questa modalità quando vuoi creare tabelle riepilogative usando i CSV
    # probabilities_*.csv prodotti dal notebook di training.
    # In questa modalità NON vengono caricati checkpoint e NON vengono rilette immagini.
    "RUN_VALIDATION_PROBABILITY_SCREENING_FROM_FILES": True,

    # Lascia False se vuoi solo creare la tabella validation dei modelli/combinazioni.
    # Metti True solo quando vuoi anche rieseguire la parte checkpoint-based:
    # estrazione embedding, late fusion e test finale.
    "RUN_CHECKPOINT_BASED_ENSEMBLE_LATE_FUSION": True,

    # Inserisci qui i path dei CSV di probabilità validation salvati dal training.
    # Formato atteso: CSV con colonne true_label_idx e prob_00_..., prob_01_..., ecc.
    # Per upsampling userai di solito probabilities_fold_fixed.csv o equivalente.
    # Per subsampling, per una validation rigorosa, usa le probabilità del fold corrispondente.
    "VALIDATION_PROBABILITY_FILES_BY_MODE": {
        # Usa SOLO i file coerenti con SAMPLING_MODE.
        # Non mischiare subsampling e upsampling nella stessa lista: hanno validation set diversi.
        "subsampling": [
        {
            "alias": "alexnet",
            "model_name": "alexnet",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/alexnet/alexnet_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "squeezenet",
            "model_name": "squeezenet",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/squeezenet/squeezenet_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet50",
            "model_name": "resnet50",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/resnet50/resnet50_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet18",
            "model_name": "resnet18",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/resnet18/resnet18_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vggnet",
            "model_name": "vggnet",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/vggnet/vggnet_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "efficientnet_b0",
            "model_name": "efficientnet_b0",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/efficientnet_b0/efficientnet_b0_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet34",
            "model_name": "resnet34",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/resnet34/resnet34_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "googlenet",
            "model_name": "googlenet",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/googlenet/googlenet_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv2",
            "model_name": "mobilenetv2",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/mobilenetv2/mobilenetv2_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv3_small",
            "model_name": "mobilenetv3_small",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/mobilenetv3_small/mobilenetv3_small_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv3_large",
            "model_name": "mobilenetv3_large",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/mobilenetv3_large/mobilenetv3_large_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "xception",
            "model_name": "xception",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/xception/xception_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_tiny_patch16_224",
            "model_name": "vit_tiny_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/vit_tiny_patch16_224/vit_tiny_patch16_224_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_small_patch16_224",
            "model_name": "vit_small_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/vit_small_patch16_224/vit_small_patch16_224_fine_tune_original_cv_plus_final_refit_full_20260506_160647" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_base_patch16_224",
            "model_name": "vit_base_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/vit_base_patch16_224/vit_base_patch16_224_fine_tune_original_cv_plus_final_refit_full_20260506_172823" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "dinov2_vit_small_patch14",
            "model_name": "dinov2_vit_small_patch14",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/dinov2_vit_small_patch14/dinov2_vit_small_patch14_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "dinov2_vit_base_patch14",
            "model_name": "dinov2_vit_base_patch14",
            "probabilities_path": str(REPO_ROOT / "results" / "subsampling/dinov2_vit_base_patch14/dinov2_vit_base_patch14_fine_tune_original_cv_plus_final_refit_full" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        ],
        "upsampling": [
        {
            "alias": "alexnet",
            "model_name": "alexnet",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/alexnet/alexnet_fine_tune_original_upsampling_full_20260508_182408" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "squeezenet",
            "model_name": "squeezenet",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/squeezenet/squeezenet_fine_tune_original_upsampling_full_20260508_171644" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet50",
            "model_name": "resnet50",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/resnet50/resnet50_fine_tune_original_upsampling_full_20260507_033033" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet18",
            "model_name": "resnet18",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/resnet18/resnet18_fine_tune_original_upsampling_full_20260507_220729" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vggnet",
            "model_name": "vggnet",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/vggnet/vggnet_fine_tune_original_upsampling_full_20260508_013542" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "efficientnet_b0",
            "model_name": "efficientnet_b0",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/efficientnet_b0/efficientnet_b0_fine_tune_original_upsampling_full_20260507_003751" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "resnet34",
            "model_name": "resnet34",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/resnet34/resnet34_fine_tune_original_upsampling_full_20260507_195111" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "googlenet",
            "model_name": "googlenet",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/googlenet/googlenet_fine_tune_original_upsampling_full_20260507_205521" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv2",
            "model_name": "mobilenetv2",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/mobilenetv2/mobilenetv2_fine_tune_original_upsampling_full_20260508_003250" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv3_small",
            "model_name": "mobilenetv3_small",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/mobilenetv3_small/mobilenetv3_small_fine_tune_original_upsampling_full_20260507_233751" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "mobilenetv3_large",
            "model_name": "mobilenetv3_large",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/mobilenetv3_large/mobilenetv3_large_fine_tune_original_upsampling_full_20260507_020800" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "xception",
            "model_name": "xception",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/xception/xception_fine_tune_original_upsampling_full_20260507_180613" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_tiny_patch16_224",
            "model_name": "vit_tiny_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/vit_tiny_patch16_224/vit_tiny_patch16_224_fine_tune_original_upsampling_full_20260506_025347" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_small_patch16_224",
            "model_name": "vit_small_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/vit_small_patch16_224/vit_small_patch16_224_fine_tune_original_upsampling_full_20260506_163405" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "vit_base_patch16_224",
            "model_name": "vit_base_patch16_224",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/vit_base_patch16_224/vit_base_patch16_224_fine_tune_original_upsampling_full_20260506_180456" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "dinov2_vit_small_patch14",
            "model_name": "dinov2_vit_small_patch14",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/dinov2_vit_small_patch14/dinov2_vit_small_patch14_fine_tune_original_upsampling_full_20260506_193719" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        {
            "alias": "dinov2_vit_base_patch14",
            "model_name": "dinov2_vit_base_patch14",
            "probabilities_path": str(REPO_ROOT / "results" / "upsampling/dinov2_vit_base_patch14/dinov2_vit_base_patch14_fine_tune_original_upsampling_full_20260506_214738" / "validation_artifacts" / "oof_probabilities.csv"),
        },
        ],
    },

    # Retrocompatibilità: puoi lasciare questa lista vuota.
    # Se la compili manualmente, ha priorità su VALIDATION_PROBABILITY_FILES_BY_MODE.
    "VALIDATION_PROBABILITY_FILES": [],

    # Screening exhaustive delle combinazioni basate su probabilità finali.
    "SCREENING_MIN_COMBO_SIZE": 2,
    "SCREENING_MAX_COMBO_SIZE": 3,
    "SCREENING_CANDIDATE_COMBINATIONS": None,
    "SCREENING_PROBABILITY_METHODS": [
        "soft_voting_uniform",
        "weighted_soft_voting_global",
        "weighted_soft_voting_per_class",
    ],
    "SCREENING_SELECTION_METRIC": "accuracy",
    "SCREENING_TOP_N_SUMMARY": 15,

    # Criteri per etichettare un ensemble come promettente.
    # La valutazione ora confronta l'ensemble sia con i modelli dentro la combinazione,
    # sia con il miglior modello singolo globale dello screening.
    "SCREENING_VERY_PROMISING_MIN_ACC_DELTA_GLOBAL": 0.003,
    "SCREENING_PROMISING_MAX_ACC_DROP_GLOBAL": 0.005,
    "SCREENING_PROMISING_MAX_F1_DROP_GLOBAL": 0.005,
    "SCREENING_PROMISING_MAX_BAL_ACC_DROP_GLOBAL": 0.007,
    "SCREENING_SELECTED_FOR_TEST_TOP_N": 8,

    "SCREENING_WEIGHT_SOURCE_METRIC": "accuracy",
    "SCREENING_EPSILON": 1e-8,

    # Dataloader per probabilità ed embedding dei modelli base.
    "IMAGE_SIZE": 224,
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 0,
    "PIN_MEMORY": False,

    # Cosa eseguire.
    "RUN_PROBABILITY_ENSEMBLES": True,
    "RUN_LATE_FUSION_EMBEDDINGS": True,

    # Combinazioni candidate.
    # Se CANDIDATE_COMBINATIONS = None, genera automaticamente tutte le combinazioni
    # tra MIN_COMBO_SIZE e MAX_COMBO_SIZE usando BASE_MODELS.
    "MIN_COMBO_SIZE": 2,
    "MAX_COMBO_SIZE": 3,
    "CANDIDATE_COMBINATIONS": None,
    # Esempio manuale:
    # "CANDIDATE_COMBINATIONS": [
    #     ["resnet18", "dinov2_small", "dinov2_base"],
    # ],

    # Metrica usata per ordinare le combinazioni in validation.
    # Valori consigliati: "accuracy", "macro_f1", "balanced_accuracy".
    "SELECTION_METRIC": "accuracy",

    # Metodi basati sulle probabilità finali dei modelli base.
    "PROBABILITY_METHODS": [
        "soft_voting_uniform",
        "weighted_soft_voting_global",
        "weighted_soft_voting_per_class",
    ],
    # Griglia per weighted soft voting globale. 0.05 è più fine ma più lenta.
    "WEIGHT_GRID_STEP": 0.10,

    # Late fusion sugli embedding concatenati.
    # "mlp" replica meglio una fusione neurale; "logistic_regression" è più veloce.
    "FUSION_CLASSIFIER": "mlp",
    "MLP_NUM_EPOCHS": 60,
    "MLP_LR": 1e-3,
    "MLP_WEIGHT_DECAY": 1e-4,
    "MLP_PATIENCE": 12,
    "MLP_HIDDEN_DIM": 512,
    "MLP_DROPOUT": 0.4,
    "MLP_BATCH_SIZE": 64,
    "LOGREG_MAX_ITER": 3000,
    "LOGREG_C": 1.0,

    # Test finale: dopo la validation, testa solo le migliori combinazioni per metodo.
    # Con valore 1 ottieni una scelta pulita: best validation per ciascun metodo.
    "TEST_TOP_N_PER_METHOD": 1,

    # Modelli già allenati da usare come candidati.
    # IMPORTANTE: aggiorna checkpoint_path con i path veri prodotti dal notebook:
    # - upsampling:  results/upsampling/<model>/<run>/models/<model>_fold_fixed.pt
    # - subsampling: results/subsampling/<model>/<run>/models/<model>_final_refit.pt
    "BASE_MODELS": [
        {
            "alias": "resnet18",
            "model_name": "resnet18",
            "checkpoint_path": str(REPO_ROOT / "results" / "upsampling/resnet18/resnet18_fine_tune_original_upsampling_full_20260507_220729" / "models" / "resnet18_fold_fixed.pt"),
        },
        {
            "alias": "dinov2_small",
            "model_name": "dinov2_vit_small_patch14",
            "checkpoint_path": str(REPO_ROOT / "results" / "upsampling/dinov2_vit_small_patch14/dinov2_vit_small_patch14_fine_tune_original_upsampling_full_20260506_193719" / "models" / "dinov2_vit_small_patch14_fold_fixed.pt"),
        },
        {
            "alias": "dinov2_base",
            "model_name": "dinov2_vit_base_patch14",
            "checkpoint_path": str(REPO_ROOT / "results" / "upsampling/dinov2_vit_base_patch14/dinov2_vit_base_patch14_fine_tune_original_upsampling_full_20260506_214738" / "models" / "dinov2_vit_base_patch14_fold_fixed.pt"),
        },

        # {
        #     "alias": "resnet18",
        #     "model_name": "resnet18",
        #     "checkpoint_path": str(REPO_ROOT / "results" / "subsampling/resnet18/resnet18_fine_tune_original_cv_plus_final_refit_full" / "models" / "resnet18_final_refit.pt"),
        # },
        # {
        #     "alias": "dinov2_small",
        #     "model_name": "dinov2_vit_small_patch14",
        #     "checkpoint_path": str(REPO_ROOT / "results" / "subsampling/dinov2_vit_small_patch14/dinov2_vit_small_patch14_fine_tune_original_cv_plus_final_refit_full" / "models" / "dinov2_vit_small_patch14_final_refit.pt"),
        # },
        # {
        #     "alias": "dinov2_base",
        #     "model_name": "dinov2_vit_base_patch14",
        #     "checkpoint_path": str(REPO_ROOT / "results" / "subsampling/dinov2_vit_base_patch14/dinov2_vit_base_patch14_fine_tune_original_cv_plus_final_refit_full" / "models" / "dinov2_vit_base_patch14_final_refit.pt"),
        # },
    ],
}


## Nota su `alias`, `model_name` e file validation

Per lo screening da CSV di probabilità:

- `alias` è il nome breve usato nelle combinazioni e nelle tabelle di ranking;
- `model_name` deve contenere solo il nome dell'architettura (`resnet18`, `dinov2_vit_base_patch14`, ecc.);
- `probabilities_path` è il CSV validation prodotto dal training (`oof_probabilities.csv`, `probabilities_fold_fixed.csv`, ecc.).

Il notebook seleziona automaticamente la lista corretta da `VALIDATION_PROBABILITY_FILES_BY_MODE` in base a `CONFIG["SAMPLING_MODE"]`, quindi non bisogna mischiare file di subsampling e upsampling nella stessa esecuzione.


In [ ]:
# Ambiente locale repo-centric: nessun mount Drive richiesto.
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Results dir: {Path(CONFIG['OUTPUT_DIR'])}")


Mounted at /content/drive
Device: cuda


In [ ]:
# ============================================================
# 3. DATASET LOCALE, SPLIT E METADATA COMPATIBILI CON TRAINING
# ============================================================

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold


def extract_tar_if_needed(tar_path: str, output_root: str):
    """
    Estrae un archivio tar solo se la cartella locale non contiene già dati.
    Serve solo per velocizzare Colab: la logica sperimentale non cambia.
    """
    tar_path = Path(tar_path)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    if not tar_path.exists():
        print(f"[SKIP] Tar non trovato: {tar_path}")
        return

    if any(output_root.iterdir()):
        print(f"[CACHE] Dataset locale già presente: {output_root}")
        return

    print(f"Estrazione tar: {tar_path} -> {output_root}")
    with tarfile.open(tar_path, "r:*") as tar:
        tar.extractall(output_root)


def prepare_local_dataset(config: dict):
    """
    Prepara eventualmente una copia locale del dataset originale.
    Non esistono più varianti background blur.
    """
    if not config.get("USE_TAR_LOCAL_DATASET", False):
        return

    extract_tar_if_needed(config["TAR_ORIGINAL_PATH"], config["DATASET_LOCAL_DIR"])

    if config["SAMPLING_MODE"] == "upsampling" and config.get("USE_TAR_AUGMENTED_UPSAMPLING", False):
        # Le immagini augmentate vengono estratte nello stesso root locale.
        extract_tar_if_needed(config["TAR_AUG_ORIGINAL_PATH"], config["DATASET_LOCAL_DIR"])


def maybe_remap_drive_paths_to_local(df: pd.DataFrame, config: dict, strict: bool = True) -> pd.DataFrame:
    """
    Sostituisce i path Drive con path locali solo quando il file locale esiste.

    Questo rende il notebook robusto:
    - se hai il tar locale, legge da /content;
    - se non hai il tar degli augmented, continua a leggere gli augmented da Drive.
    """
    if "image_path" not in df.columns:
        raise ValueError("Colonna 'image_path' mancante nei metadata.")

    df = df.copy()
    if not config.get("USE_TAR_LOCAL_DATASET", False):
        if strict:
            missing = ~df["image_path"].apply(lambda p: Path(p).exists())
            if missing.any():
                raise FileNotFoundError(f"Mancano {int(missing.sum())} immagini su Drive.")
        return df

    drive_root = str(config["DATASET_DRIVE_DIR"])
    local_root = str(config["DATASET_LOCAL_DIR"])

    remapped_paths = []
    for p in df["image_path"].astype(str):
        local_p = p.replace(drive_root, local_root, 1)
        if Path(local_p).exists():
            remapped_paths.append(local_p)
        else:
            remapped_paths.append(p)

    df["image_path"] = remapped_paths

    if strict:
        missing = ~df["image_path"].apply(lambda p: Path(p).exists())
        if missing.any():
            print("Prime immagini mancanti:")
            print(df.loc[missing, "image_path"].head(10).to_string(index=False))
            raise FileNotFoundError(
                f"Mancano {int(missing.sum())} immagini su {len(df)}. "
                "Controlla path Drive, tar locale o cartella augmented."
            )

    return df


def scan_original_dataset(config: dict):
    """
    Ricostruisce df_all e label mapping esattamente come nel notebook training:
    classi ordinate alfabeticamente, immagini ordinate per path.
    """
    dataset_root = Path(config["DATASET_DRIVE_DIR"]) / "Comics"
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset originale non trovato: {dataset_root}")

    valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
    records = []

    class_dirs = [p for p in dataset_root.iterdir() if p.is_dir()]
    class_dirs = sorted(class_dirs, key=lambda p: p.name.lower())

    for class_dir in class_dirs:
        class_name = class_dir.name
        for img_path in sorted(class_dir.rglob("*"), key=lambda p: str(p).lower()):
            if img_path.is_file() and img_path.suffix.lower() in valid_exts:
                records.append({
                    "image_path": str(img_path),
                    "image_name": img_path.name,
                    "class_name": class_name,
                })

    df_all = pd.DataFrame(records)
    if df_all.empty:
        raise ValueError(f"Nessuna immagine trovata in {dataset_root}")

    valid_classes = sorted(df_all["class_name"].unique())
    label_map = {class_name: idx for idx, class_name in enumerate(valid_classes)}
    inv_label_map = {idx: class_name for class_name, idx in label_map.items()}

    df_all["label"] = df_all["class_name"].map(label_map).astype(int)

    class_mapping_df = pd.DataFrame({
        "class_name": valid_classes,
        "label": [label_map[c] for c in valid_classes],
    })

    return df_all, class_mapping_df, label_map, inv_label_map


def load_upsampling_splits(config: dict, df_all: pd.DataFrame):
    """
    Ricostruisce lo split 80/10/10 del preset train_upsampling_original.
    Il train viene letto dal manifest prodotto dal notebook training, perché contiene
    originali + immagini augmentate. Nel protocollo upsampling il train finale coincide
    con il train usato per il classificatore di late fusion.
    """
    seed = int(config["SEED"])
    test_size = float(config["TEST_SIZE"])

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_val_idx, test_idx = next(sss_test.split(df_all, df_all["label"]))

    df_train_val = df_all.iloc[train_val_idx].reset_index(drop=True)
    test_df = df_all.iloc[test_idx].reset_index(drop=True)

    # Come nel notebook di training: 0.1111 del train_val ≈ 10% del dataset totale.
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.1111, random_state=seed)
    train_idx, val_idx = next(sss_val.split(df_train_val, df_train_val["label"]))

    train_original_df = df_train_val.iloc[train_idx].reset_index(drop=True)
    val_df = df_train_val.iloc[val_idx].reset_index(drop=True)

    aug_dir = Path(config["DATASET_DRIVE_DIR"]) / (
        f"augmented_train_only_original_"
        f"seed{seed}_"
        f"test{str(test_size).replace('.', 'p')}"
    )
    manifest_path = aug_dir / "train_upsampling_manifest.csv"

    if manifest_path.exists():
        train_df = pd.read_csv(manifest_path)
        split_info = {
            "mode": "upsampling",
            "train_source": str(manifest_path),
            "val_source": "reconstructed_from_seed",
            "test_source": "reconstructed_from_seed",
            "final_train_source": str(manifest_path),
        }
    else:
        raise FileNotFoundError(
            f"Manifest upsampling non trovato: {manifest_path}\n"
            "Esegui prima il preset train_upsampling_original nel notebook training, "
            "oppure crea la cartella augmented_train_only_original con il manifest."
        )

    for df in [train_df, val_df, test_df]:
        if "label" in df.columns:
            df["label"] = df["label"].astype(int)

    final_train_df = train_df.copy()
    return train_df, val_df, test_df, final_train_df, split_info


def load_subsampling_splits(config: dict, df_all: pd.DataFrame):
    """
    Ricostruisce il protocollo subsampling:
    dataset bilanciato a min_class_count, split test, poi assegnazione fold CV.

    Restituisce due concetti distinti:
    - train_df/val_df: fold usati per validare combinazioni con checkpoint fold-specific;
    - final_train_df: intero CV pool, usato per ri-addestrare il classificatore finale
      di late fusion sui checkpoint final_refit prima del test separato.
    """
    seed = int(config["SEED"])
    test_size = float(config["TEST_SIZE"])
    n_splits = int(config["N_SPLITS"])
    val_fold = int(config["VAL_FOLD_FOR_SUBSAMPLING"])

    min_class_count = df_all["class_name"].value_counts().min()

    df_all_balanced = (
        df_all.groupby("class_name", group_keys=False, sort=False)
        .sample(n=min_class_count, random_state=seed)
        .reset_index(drop=True)
    )

    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(sss.split(df_all_balanced, df_all_balanced["label"]))

    cv_pool_df = df_all_balanced.iloc[train_idx].reset_index(drop=True)
    test_df = df_all_balanced.iloc[test_idx].reset_index(drop=True)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cv_pool_df["fold"] = -1

    for fold_idx, (_, val_idx) in enumerate(skf.split(cv_pool_df, cv_pool_df["label"])):
        cv_pool_df.loc[val_idx, "fold"] = fold_idx

    if not (0 <= val_fold < n_splits):
        raise ValueError(f"VAL_FOLD_FOR_SUBSAMPLING deve essere tra 0 e {n_splits - 1}")

    train_df = cv_pool_df[cv_pool_df["fold"] != val_fold].reset_index(drop=True)
    val_df = cv_pool_df[cv_pool_df["fold"] == val_fold].reset_index(drop=True)
    final_train_df = cv_pool_df.reset_index(drop=True)

    split_info = {
        "mode": "subsampling",
        "train_source": f"reconstructed_cv_pool_fold_not_{val_fold}",
        "val_source": f"reconstructed_cv_pool_fold_{val_fold}",
        "final_train_source": "reconstructed_full_cv_pool_train_plus_validation",
        "test_source": "reconstructed_test_split",
        "val_fold": val_fold,
        "min_class_count": int(min_class_count),
    }

    return train_df, val_df, test_df, final_train_df, split_info


def get_label_column(df: pd.DataFrame) -> str:
    """Rende esplicita la colonna label usata nei metadata."""
    for candidate in ["label", "true_label_idx", "class_idx"]:
        if candidate in df.columns:
            return candidate
    raise ValueError("Nessuna colonna label trovata. Attese: label, true_label_idx o class_idx.")


def load_late_fusion_metadata(config: dict):
    """
    Carica/ricostruisce gli split in modo compatibile con training.
    Non richiede più results/<mode>/generated_metadata.

    Restituisce anche final_train_df, necessario nel subsampling per il test finale:
    dopo aver scelto la combinazione su un fold, la late fusion viene ri-addestrata
    sugli embedding del CV pool completo prodotti dai checkpoint final_refit.
    """
    df_all, class_mapping_df, label_map, inv_label_map = scan_original_dataset(config)
    num_classes = int(len(class_mapping_df))

    mode = config["SAMPLING_MODE"]
    if mode == "upsampling":
        train_df, val_df, test_df, final_train_df, split_info = load_upsampling_splits(config, df_all)
    elif mode == "subsampling":
        train_df, val_df, test_df, final_train_df, split_info = load_subsampling_splits(config, df_all)
    else:
        raise ValueError("SAMPLING_MODE deve essere 'upsampling' oppure 'subsampling'.")

    # Usa /content quando i file locali esistono, altrimenti resta su Drive.
    train_df = maybe_remap_drive_paths_to_local(train_df, config)
    val_df = maybe_remap_drive_paths_to_local(val_df, config)
    test_df = maybe_remap_drive_paths_to_local(test_df, config)
    final_train_df = maybe_remap_drive_paths_to_local(final_train_df, config)

    print("Metadata compatibili caricati/ricostruiti")
    print(
        "Train:", len(train_df),
        "| Val:", len(val_df),
        "| Final train:", len(final_train_df),
        "| Test:", len(test_df),
        "| Classi:", num_classes,
    )
    print("Split info:", split_info)

    return train_df, val_df, test_df, final_train_df, class_mapping_df, split_info, num_classes


class ComicsImageDataset(Dataset):
    """Dataset deterministico per estrarre embedding: niente augmentation online."""
    def __init__(self, df: pd.DataFrame, image_size: int):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.label_col = get_label_column(self.df)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["image_path"]) as img:
            img = img.convert("RGB")
            x = self.transform(img)
        y = int(row[self.label_col])
        return x, y


def make_image_loader(df: pd.DataFrame, config: dict):
    dataset = ComicsImageDataset(df, image_size=int(config["IMAGE_SIZE"]))
    return DataLoader(
        dataset,
        batch_size=int(config["BATCH_SIZE"]),
        shuffle=False,
        num_workers=int(config["NUM_WORKERS"]),
        pin_memory=bool(config["PIN_MEMORY"]),
    )


In [ ]:
# ============================================================
# 4. MODELLI BASE GIÀ ALLENATI: BUILD, LOAD, PROBABILITÀ, EMBEDDING
# ============================================================

def build_base_model(model_name: str, num_classes: int, image_size: int):
    """
    Ricostruisce l'architettura dei modelli già allenati.
    Non esegue training: serve solo a caricare checkpoint, probabilità ed embedding.
    """
    model_name = model_name.lower()

    if model_name == "alexnet":
        model = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    elif model_name == "vggnet":
        model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    elif model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "resnet34":
        model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "resnet50":
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "googlenet":
        model = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1, aux_logits=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        if model.aux1 is not None:
            model.aux1.fc2 = nn.Linear(model.aux1.fc2.in_features, num_classes)
        if model.aux2 is not None:
            model.aux2.fc2 = nn.Linear(model.aux2.fc2.in_features, num_classes)
        model.aux_logits = False

    elif model_name == "mobilenetv2":
        model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenetv3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    elif model_name == "mobilenetv3_large":
        model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "squeezenet":
        model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)

    elif model_name == "xception":
        model = timm.create_model("xception", pretrained=True, num_classes=num_classes)

    elif model_name == "dinov2_vit_small_patch14":
        model = timm.create_model(
            "vit_small_patch14_dinov2.lvd142m",
            pretrained=True,
            num_classes=num_classes,
            img_size=image_size,
        )

    elif model_name == "dinov2_vit_base_patch14":
        model = timm.create_model(
            "vit_base_patch14_dinov2.lvd142m",
            pretrained=True,
            num_classes=num_classes,
            img_size=image_size,
        )

    elif model_name in {"vit_tiny_patch16_224", "vit_small_patch16_224", "vit_base_patch16_224"}:
        model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    else:
        raise ValueError(f"Modello base non supportato in ensemble/late fusion: {model_name}")

    for param in model.parameters():
        param.requires_grad = False

    return model


def load_checkpoint_into_model(model: nn.Module, checkpoint_path: str, device):
    """Carica pesi salvati dal notebook di training."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint non trovato: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    clean_state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}
    missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)

    print(f"Checkpoint caricato: {checkpoint_path.name} | missing={len(missing)} | unexpected={len(unexpected)}")
    if missing:
        print("Prime missing keys:", missing[:5])
    if unexpected:
        print("Prime unexpected keys:", unexpected[:5])

    return model


def get_classifier_head(model: nn.Module, model_name: str = None):
    """
    Restituisce il classificatore finale.
    Il forward hook cattura il suo input, cioè l'embedding prima della head.
    """
    model_name = (model_name or "").lower()

    if model_name == "squeezenet" and hasattr(model, "classifier"):
        # In SqueezeNet la vera proiezione a classi è Conv2d in classifier[1].
        return model.classifier[1]

    if hasattr(model, "fc") and isinstance(model.fc, nn.Module):
        return model.fc

    if hasattr(model, "get_classifier"):
        head = model.get_classifier()
        if isinstance(head, nn.Module):
            return head

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Module):
        if isinstance(model.classifier, nn.Sequential):
            # Prende l'ultimo Linear/Conv2d, non dropout/pooling finali.
            for layer in reversed(model.classifier):
                if isinstance(layer, (nn.Linear, nn.Conv2d)):
                    return layer
            return model.classifier[-1]
        return model.classifier

    raise ValueError("Classifier head non trovata. Aggiungi un caso specifico per questo modello.")


def logits_from_model_output(output):
    """Normalizza output diversi: Tensor, tuple/list o oggetti con attributo logits."""
    if isinstance(output, (tuple, list)):
        output = output[0]
    if hasattr(output, "logits"):
        output = output.logits
    return output


@torch.no_grad()
def extract_outputs(model: nn.Module, model_name: str, loader: DataLoader, device):
    """
    In un solo passaggio sul dataset estrae:
    - embedding prima della classification head;
    - probabilità finali softmax del modello base;
    - label vere.
    """
    model.eval()
    model.to(device)

    head = get_classifier_head(model, model_name=model_name)
    captured = []

    def hook_fn(module, inputs, output):
        features = inputs[0].detach()

        if features.ndim == 4:
            # CNN: [B, C, H, W] -> Global Average Pooling -> [B, C].
            features = F.adaptive_avg_pool2d(features, (1, 1))
            features = torch.flatten(features, 1)
        elif features.ndim == 3:
            # Transformer: [B, token, D]. Se arriva qui, usa CLS token.
            features = features[:, 0, :]
        elif features.ndim != 2:
            features = torch.flatten(features, 1)

        captured.append(features.cpu())

    handle = head.register_forward_hook(hook_fn)

    labels = []
    probabilities = []

    for images, y in tqdm(loader, desc=f"Output {model_name}", leave=False):
        images = images.to(device, non_blocking=True)
        output = model(images)
        logits = logits_from_model_output(output)
        prob = torch.softmax(logits, dim=1)

        probabilities.append(prob.cpu())
        labels.extend(y.cpu().numpy().tolist())

    handle.remove()

    X = torch.cat(captured, dim=0).numpy().astype(np.float32)
    P = torch.cat(probabilities, dim=0).numpy().astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)

    return X, P, y


def infer_subsampling_fold_checkpoint_path(run_cfg: dict, config: dict) -> Path:
    """
    Inferisce il checkpoint fold-specific prodotto dal training.
    Caso standard:
    .../models/<model>_final_refit.pt  ->  .../models/<model>_fold_0.pt
    """
    val_fold = int(config["VAL_FOLD_FOR_SUBSAMPLING"])

    explicit = run_cfg.get("fold_checkpoint_path")
    if explicit:
        return Path(str(explicit).format(fold=val_fold))

    p = Path(run_cfg["checkpoint_path"])
    model_name = run_cfg["model_name"]

    # Se l'utente ha già passato un checkpoint fold-specific, lo usa direttamente.
    if re.search(r"_fold_\d+\.pt$", p.name):
        return p

    candidates = []
    if "final_refit" in p.name:
        candidates.append(p.with_name(p.name.replace("final_refit", f"fold_{val_fold}")))
    candidates.append(p.parent / f"{model_name}_fold_{val_fold}.pt")
    candidates.append(p.parent / f"{model_name}_fold_{val_fold}.pth")

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Restituisce il candidato standard: l'errore esplicito arriverà al load.
    return candidates[0]


def infer_subsampling_final_refit_checkpoint_path(run_cfg: dict) -> Path:
    """
    Inferisce il checkpoint final_refit prodotto dal training.
    Caso standard:
    .../models/<model>_fold_0.pt  ->  .../models/<model>_final_refit.pt
    """
    explicit = run_cfg.get("final_refit_checkpoint_path")
    if explicit:
        return Path(explicit)

    p = Path(run_cfg["checkpoint_path"])
    model_name = run_cfg["model_name"]

    if "final_refit" in p.name:
        return p

    if re.search(r"_fold_\d+\.pt$", p.name):
        return p.with_name(re.sub(r"_fold_\d+\.pt$", "_final_refit.pt", p.name))

    candidate = p.parent / f"{model_name}_final_refit.pt"
    return candidate


def resolve_checkpoint_path_for_split(run_cfg: dict, split_name: str, config: dict) -> Path:
    """
    Decide quale checkpoint usare per ogni split.

    Upsampling:
    - stesso checkpoint fold_fixed per train/val/test.

    Subsampling:
    - train e val: checkpoint fold-specific, perché la validation non deve essere vista dal modello base;
    - final_train e test: checkpoint final_refit, perché il test finale usa il modello riaddestrato sul CV pool.
    """
    mode = str(config["SAMPLING_MODE"]).lower()
    if mode != "subsampling":
        return Path(run_cfg["checkpoint_path"])

    if split_name in {"train", "val"}:
        return infer_subsampling_fold_checkpoint_path(run_cfg, config)

    if split_name in {"final_train", "test"}:
        return infer_subsampling_final_refit_checkpoint_path(run_cfg)

    raise ValueError(f"Split non riconosciuto per checkpoint resolution: {split_name}")


def extract_or_load_model_outputs(run_cfg: dict, split_name: str, df_split: pd.DataFrame, output_dir: Path, config: dict, num_classes: int):
    """Usa cache .npy se disponibile, altrimenti estrae embedding e probabilità dal checkpoint."""
    alias = run_cfg["alias"]
    model_name = run_cfg["model_name"]

    feature_dir = output_dir / "base_model_outputs" / alias
    feature_dir.mkdir(parents=True, exist_ok=True)

    features_path = feature_dir / f"{split_name}_features.npy"
    prob_path = feature_dir / f"{split_name}_probabilities.npy"
    labels_path = feature_dir / f"{split_name}_labels.npy"
    metadata_path = feature_dir / f"{split_name}_metadata.csv"

    use_cache = (
        features_path.exists()
        and prob_path.exists()
        and labels_path.exists()
        and not config["FORCE_REEXTRACT_OUTPUTS"]
    )

    if use_cache:
      print(f"[CACHE] {alias} | {split_name}")
      return {
          "features": np.load(features_path).astype(np.float32),
          "probabilities": normalize_probabilities(np.load(prob_path)).astype(np.float32),
          "labels": np.load(labels_path).astype(np.int64),
      }

    loader = make_image_loader(df_split, config)

    model = build_base_model(
        model_name=model_name,
        num_classes=num_classes,
        image_size=int(config["IMAGE_SIZE"]),
    )
    checkpoint_path = resolve_checkpoint_path_for_split(run_cfg, split_name, config)
    model = load_checkpoint_into_model(model, checkpoint_path, device)

    features, probabilities, labels = extract_outputs(model, model_name, loader, device)
    probabilities = normalize_probabilities(probabilities).astype(np.float32)

    np.save(features_path, features)
    np.save(prob_path, probabilities)
    np.save(labels_path, labels)

    meta = df_split.reset_index(drop=True).copy()
    meta["ensemble_late_fusion_label"] = labels
    meta.to_csv(metadata_path, index=False, encoding="utf-8-sig")

    print(f"Salvati output {alias} | {split_name}: features={features.shape}, prob={probabilities.shape}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "features": features,
        "probabilities": probabilities,
        "labels": labels,
    }


In [ ]:
# ============================================================
# 5. CLASSIFICATORE FINALE E METRICHE
# ============================================================
def normalize_probabilities(y_prob, eps: float = 1e-12):
    """
    Converte qualsiasi matrice di score/probabilità in una matrice probabilistica valida.

    Serve per evitare warning/errori in log_loss ed ECE:
    - valori negativi o NaN vengono gestiti;
    - ogni riga viene rinormalizzata per sommare a 1;
    - righe patologiche vengono sostituite con distribuzione uniforme.
    """
    y_prob = np.asarray(y_prob, dtype=np.float64)

    # Gestione NaN/inf e clipping numerico.
    y_prob = np.nan_to_num(y_prob, nan=0.0, posinf=1.0, neginf=0.0)
    y_prob = np.clip(y_prob, eps, None)

    row_sums = y_prob.sum(axis=1, keepdims=True)

    # Se qualche riga ha somma nulla o non valida, usa distribuzione uniforme.
    bad_rows = (~np.isfinite(row_sums)) | (row_sums <= eps)
    if np.any(bad_rows):
        num_classes = y_prob.shape[1]
        y_prob[bad_rows[:, 0], :] = 1.0 / num_classes
        row_sums = y_prob.sum(axis=1, keepdims=True)

    y_prob = y_prob / row_sums
    return y_prob

def expected_calibration_error(y_true, y_prob, n_bins: int = 10) -> float:
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct = (predictions == y_true).astype(float)

    ece = 0.0
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    for start, end in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > start) & (confidences <= end)
        if mask.any():
            ece += (mask.mean()) * abs(correct[mask].mean() - confidences[mask].mean())

    return float(ece)


def compute_metrics(y_true, y_prob, num_classes: int):
    """
    Calcola metriche di classificazione multi-classe.

    Nota importante:
    y_prob viene sempre rinormalizzato prima di log_loss/ECE.
    Questo evita warning sklearn quando le righe non sommano esattamente a 1,
    cosa che può succedere con weighted soft voting, pesi per classe o float32.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = normalize_probabilities(y_prob)

    y_pred = np.argmax(y_prob, axis=1)
    labels = list(range(num_classes))

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "expected_calibration_error": expected_calibration_error(y_true, y_prob),
        "num_samples": int(len(y_true)),
    }

    try:
        metrics["log_loss"] = float(log_loss(y_true, y_prob, labels=labels))
    except Exception as exc:
        print(f"[WARN] log_loss non calcolabile: {exc}")
        metrics["log_loss"] = None

    for k in [3, 5]:
        if num_classes >= k:
            topk = np.argsort(y_prob, axis=1)[:, -k:]
            metrics[f"top{k}_accuracy"] = float(
                np.mean([yt in row for yt, row in zip(y_true, topk)])
            )
        else:
            metrics[f"top{k}_accuracy"] = None

    return metrics, y_pred


class LateFusionFeatureDataset(Dataset):
    """Dataset tabellare: ogni riga è la concatenazione degli embedding dei modelli base."""
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = features.astype(np.float32)
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.from_numpy(self.features[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


class LateFusionMLP(nn.Module):
    """Classificatore finale leggero sulle feature concatenate."""
    def __init__(self, input_dim: int, num_classes: int, hidden_dim: int = 512, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def predict_mlp_probabilities(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int, device):
    dataset = LateFusionFeatureDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    all_prob = []

    for features, _ in loader:
        features = features.to(device)
        logits = model(features)
        prob = torch.softmax(logits, dim=1)
        all_prob.append(prob.cpu().numpy())

    return normalize_probabilities(np.concatenate(all_prob, axis=0)).astype(np.float32)


def fit_mlp_classifier(X_train, y_train, X_val, y_val, config: dict, output_dir: Path, num_classes: int):
    """Addestra solo il classificatore finale di fusione, non le reti base."""
    train_dataset = LateFusionFeatureDataset(X_train, y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(config["MLP_BATCH_SIZE"]),
        shuffle=True,
        num_workers=0,
    )

    model = LateFusionMLP(
        input_dim=X_train.shape[1],
        num_classes=num_classes,
        hidden_dim=int(config["MLP_HIDDEN_DIM"]),
        dropout=float(config["MLP_DROPOUT"]),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config["MLP_LR"]),
        weight_decay=float(config["MLP_WEIGHT_DECAY"]),
    )

    best_state = None
    best_val_metric = -1.0
    best_epoch = -1
    patience_left = int(config["MLP_PATIENCE"])
    history = []
    selection_metric = config.get("SELECTION_METRIC", "accuracy")

    for epoch in range(1, int(config["MLP_NUM_EPOCHS"]) + 1):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * features.size(0)

        train_loss = epoch_loss / len(train_dataset)

        val_prob = predict_mlp_probabilities(
            model=model,
            X=X_val,
            y=y_val,
            batch_size=128,
            device=device,
        )
        val_metrics, _ = compute_metrics(y_val, val_prob, num_classes)
        score = val_metrics[selection_metric]

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"loss={train_loss:.4f} | "
            f"internal_val_acc={val_metrics['accuracy']:.4f} | "
            f"internal_val_f1={val_metrics['macro_f1']:.4f}"
        )

        if score > best_val_metric:
            best_val_metric = score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = int(config["MLP_PATIENCE"])

            torch.save(
                {
                    "model_state_dict": best_state,
                    "input_dim": int(X_train.shape[1]),
                    "num_classes": int(num_classes),
                    "best_epoch": int(best_epoch),
                    f"best_val_{selection_metric}": float(best_val_metric),
                },
                output_dir / "late_fusion_mlp_best.pt",
            )
        else:
            patience_left -= 1

        if patience_left <= 0:
            print("Early stopping del classificatore finale.")
            break

    if best_state is None:
        raise RuntimeError("MLP late fusion non ha prodotto uno stato valido.")

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    history_df.to_csv(output_dir / "late_fusion_mlp_history.csv", index=False, encoding="utf-8-sig")

    return model, {
        "classifier": "mlp",
        "best_epoch": int(best_epoch),
        f"best_val_{selection_metric}": float(best_val_metric),
    }


def fit_logistic_regression_classifier(X_train, y_train, config: dict, output_dir: Path):
    """Alternativa non neurale: Logistic Regression sulle feature concatenate."""
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=float(config["LOGREG_C"]),
            max_iter=int(config["LOGREG_MAX_ITER"]),
            multi_class="auto",
            n_jobs=-1,
        ),
    )
    classifier.fit(X_train, y_train)

    with open(output_dir / "late_fusion_logistic_regression.pkl", "wb") as f:
        pickle.dump(classifier, f)

    return classifier, {"classifier": "logistic_regression"}


def predict_classifier_probabilities(classifier, X: np.ndarray, y: np.ndarray, config: dict):
    if isinstance(classifier, nn.Module):
        prob = predict_mlp_probabilities(
            model=classifier,
            X=X,
            y=y,
            batch_size=128,
            device=device,
        )
        return normalize_probabilities(prob).astype(np.float32)

    if hasattr(classifier, "predict_proba"):
        prob = classifier.predict_proba(X)
        return normalize_probabilities(prob).astype(np.float32)

    raise ValueError("Il classificatore finale non espone predict_proba.")


In [ ]:
# ============================================================
# 5B. SCREENING VALIDATION DA FILE DI PROBABILITÀ
# ============================================================
# Questa sezione serve per creare una tabella riepilogativa dei modelli/combinazioni
# usando i CSV di probabilità salvati dal notebook di training.
#
# Vantaggio: è molto più veloce della modalità checkpoint-based, perché non ricarica
# immagini e non ricalcola forward pass. Lavora direttamente su:
# - true_label_idx
# - prob_00_..., prob_01_..., ..., prob_45_...


def _screening_placeholder_path(path_value: str) -> bool:
    path_value = str(path_value)
    return (
        not path_value
        or "INSERISCI_PATH" in path_value
        or path_value.strip().lower() in {"none", "todo", "path"}
    )


def detect_probability_columns(df: pd.DataFrame):
    """Trova e ordina le colonne prob_XX_* generate dal notebook training."""
    prob_cols = []
    for col in df.columns:
        match = re.match(r"^prob_(\d+)(?:_|$)", str(col))
        if match:
            prob_cols.append((int(match.group(1)), col))

    if not prob_cols:
        raise ValueError(
            "Nessuna colonna di probabilità trovata. "
            "Il CSV deve contenere colonne tipo prob_00_nomeClasse, prob_01_nomeClasse, ..."
        )

    prob_cols = [col for _, col in sorted(prob_cols, key=lambda x: x[0])]
    return prob_cols


def choose_alignment_key(dfs):
    """
    Sceglie una chiave stabile per allineare i CSV dei diversi modelli.
    Preferisce image_path; se non presente usa image_name. Se nessuna chiave è disponibile,
    il notebook assume che le righe siano già nello stesso ordine.
    """
    candidates = ["image_path", "relative_path", "image_name"]
    for key in candidates:
        if all(key in df.columns for df in dfs):
            return key
    return None


def load_validation_probability_files(config: dict):
    """
    Carica i CSV di probabilità validation dei singoli modelli e li allinea sugli stessi campioni.
    Ogni modello deve avere le stesse immagini e le stesse classi nello stesso validation set.

    Formato consigliato di ogni elemento:
        {"alias": "resnet18", "model_name": "resnet18", "probabilities_path": ".../oof_probabilities.csv"}

    Nota: se VALIDATION_PROBABILITY_FILES è vuoto, usa automaticamente
    VALIDATION_PROBABILITY_FILES_BY_MODE[CONFIG["SAMPLING_MODE"]].
    """
    files_cfg = config.get("VALIDATION_PROBABILITY_FILES") or []

    if not files_cfg:
        by_mode = config.get("VALIDATION_PROBABILITY_FILES_BY_MODE", {}) or {}
        sampling_mode = config.get("SAMPLING_MODE")
        files_cfg = by_mode.get(sampling_mode, [])

    # Protezione da errore frequente: VALIDATION_PROBABILITY_FILES = [[{...}, {...}]].
    if len(files_cfg) == 1 and isinstance(files_cfg[0], list):
        files_cfg = files_cfg[0]

    if not files_cfg:
        raise ValueError(
            "Nessun file di probabilità trovato. Compila VALIDATION_PROBABILITY_FILES "
            "oppure VALIDATION_PROBABILITY_FILES_BY_MODE[SAMPLING_MODE]."
        )

    if len(files_cfg) < 2:
        raise ValueError("Per valutare ensemble servono almeno due file di probabilità.")

    loaded = []
    for item in files_cfg:
        alias = item.get("alias") or item.get("name")
        model_name = item.get("model_name") or alias
        path = item.get("probabilities_path") or item.get("validation_probabilities_path")

        if not alias:
            raise ValueError(f"Elemento non valido senza alias/name: {item}")
        if not path:
            raise ValueError(
                f"Path mancante per alias={alias}. Usa la chiave probabilities_path "
                "oppure validation_probabilities_path."
            )

        if _screening_placeholder_path(path):
            raise ValueError(
                f"Path non compilato per alias={alias}. "
                "Sostituisci INSERISCI_PATH... con il path reale del CSV probabilities_*.csv."
            )

        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"File probabilità non trovato per {alias}: {path}")

        df = pd.read_csv(path)
        prob_cols = detect_probability_columns(df)

        if "true_label_idx" not in df.columns:
            raise ValueError(f"{path} non contiene la colonna true_label_idx.")

        y_prob = df[prob_cols].astype(float).to_numpy()
        row_sums = y_prob.sum(axis=1, keepdims=True)

        # Protezione: se il CSV contiene piccole imprecisioni numeriche, rinormalizzo.
        # Se una riga ha somma zero, viene segnalato un errore perché le probabilità non sono valide.
        if np.any(row_sums <= 0):
            raise ValueError(f"{path} contiene righe con somma probabilità <= 0.")
        y_prob = y_prob / row_sums

        loaded.append({
            "alias": alias,
            "model_name": model_name,
            "path": str(path),
            "df": df,
            "prob_cols": prob_cols,
            "y_prob": y_prob,
            "y_true": df["true_label_idx"].astype(int).to_numpy(),
        })

    key = choose_alignment_key([x["df"] for x in loaded])
    base = loaded[0]
    base_df = base["df"].copy()
    base_y_true = base["y_true"]
    num_classes = len(base["prob_cols"])

    aligned_probs = {}
    metadata_cols = [
        c for c in ["image_path", "image_name", "true_label_idx", "true_class_name"]
        if c in base_df.columns
    ]
    metadata = base_df[metadata_cols].copy() if metadata_cols else pd.DataFrame(index=range(len(base_df)))

    if key is None:
        # Fallback: stesso ordine di righe.
        for item in loaded:
            if len(item["df"]) != len(base_df):
                raise ValueError(
                    "I CSV non hanno lo stesso numero di righe e non esiste una chiave "
                    "image_path/image_name per allinearli."
                )
            if len(item["prob_cols"]) != num_classes:
                raise ValueError(f"Numero classi diverso per {item['alias']}.")
            if not np.array_equal(item["y_true"], base_y_true):
                raise ValueError(
                    f"Le true_label_idx di {item['alias']} non coincidono con quelle del primo modello."
                )
            aligned_probs[item["alias"]] = item["y_prob"]
    else:
        # Allineamento esplicito rispetto al primo modello.
        base_keys = base_df[key].astype(str).tolist()
        if pd.Series(base_keys).duplicated().any():
            raise ValueError(
                f"La chiave {key} contiene duplicati. Usa image_path se disponibile, "
                "oppure verifica i CSV."
            )

        for item in loaded:
            df_item = item["df"].copy()
            if len(item["prob_cols"]) != num_classes:
                raise ValueError(f"Numero classi diverso per {item['alias']}.")

            df_item[key] = df_item[key].astype(str)
            if df_item[key].duplicated().any():
                raise ValueError(f"La chiave {key} contiene duplicati nel file {item['path']}.")

            df_indexed = df_item.set_index(key).reindex(base_keys)
            if df_indexed["true_label_idx"].isna().any():
                missing = df_indexed["true_label_idx"].isna().sum()
                raise ValueError(f"{item['alias']} non contiene {missing} immagini del validation set base.")

            y_true_item = df_indexed["true_label_idx"].astype(int).to_numpy()
            if not np.array_equal(y_true_item, base_y_true):
                raise ValueError(
                    f"Le true_label_idx di {item['alias']} non coincidono dopo allineamento su {key}."
                )

            y_prob = df_indexed[item["prob_cols"]].astype(float).to_numpy()
            y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)
            aligned_probs[item["alias"]] = y_prob

    alias_to_model_name = {item["alias"]: item["model_name"] for item in loaded}
    alias_to_path = {item["alias"]: item["path"] for item in loaded}

    print(f"Caricati {len(aligned_probs)} modelli su {len(base_y_true)} campioni validation.")
    print(f"Numero classi: {num_classes}. Chiave allineamento: {key or 'ordine righe'}")

    return {
        "y_true": base_y_true,
        "model_probs": aligned_probs,
        "metadata": metadata,
        "num_classes": num_classes,
        "alias_to_model_name": alias_to_model_name,
        "alias_to_path": alias_to_path,
        "alignment_key": key,
    }


def compute_screening_metrics(y_true, y_prob, num_classes: int):
    """
    Metriche pensate per ranking validation:
    - accuracy / balanced accuracy / macro F1 per performance classificativa;
    - log-loss ed ECE per qualità probabilistica;
    - top-k e mean true class rank per capire se la classe corretta è comunque alta.
    """
    from sklearn.metrics import precision_score, recall_score

    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)
    y_pred = np.argmax(y_prob, axis=1)

    labels = list(range(num_classes))

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "log_loss": float(log_loss(y_true, y_prob, labels=labels)),
        "expected_calibration_error": expected_calibration_error(y_true, y_prob),
        "num_samples": int(len(y_true)),
        "num_classes": int(num_classes),
    }

    order = np.argsort(-y_prob, axis=1)
    true_ranks = np.argmax(order == y_true[:, None], axis=1) + 1

    metrics["mean_true_class_rank"] = float(np.mean(true_ranks))
    metrics["mean_reciprocal_rank"] = float(np.mean(1.0 / true_ranks))

    for k in [3, 5]:
        if num_classes >= k:
            metrics[f"top{k}_acc"] = float(np.mean([y_true[i] in order[i, :k] for i in range(len(y_true))]))
        else:
            metrics[f"top{k}_acc"] = None

    return metrics


def compute_single_model_screening(bundle: dict):
    rows = []
    y_true = bundle["y_true"]
    num_classes = bundle["num_classes"]

    for alias, probs in bundle["model_probs"].items():
        row = {
            "candidate_type": "single_model",
            "alias": alias,
            "model_name": bundle["alias_to_model_name"].get(alias, alias),
            "probabilities_path": bundle["alias_to_path"].get(alias),
        }
        row.update(compute_screening_metrics(y_true, probs, num_classes))
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values("accuracy", ascending=False).reset_index(drop=True)
    df.insert(0, "rank_by_accuracy", range(1, len(df) + 1))
    return df


def build_screening_candidate_combinations(config: dict, aliases):
    manual = config.get("SCREENING_CANDIDATE_COMBINATIONS")
    if manual:
        combos = [tuple(combo) for combo in manual]
    else:
        min_size = int(config.get("SCREENING_MIN_COMBO_SIZE", 2))
        max_size = min(int(config.get("SCREENING_MAX_COMBO_SIZE", len(aliases))), len(aliases))
        combos = []
        for size in range(min_size, max_size + 1):
            combos.extend(itertools.combinations(aliases, size))

    unknown = sorted({alias for combo in combos for alias in combo if alias not in aliases})
    if unknown:
        raise ValueError(f"Alias non presenti nei file caricati: {unknown}")

    return combos


def normalize_weights(weights, eps: float = 1e-8):
    weights = np.asarray(weights, dtype=float)
    weights = np.maximum(weights, eps)
    return weights / weights.sum()


def global_metric_weights(combo, single_df: pd.DataFrame, metric: str = "accuracy", eps: float = 1e-8):
    metric_lookup = single_df.set_index("alias")[metric].to_dict()
    raw = np.array([metric_lookup[a] for a in combo], dtype=float)
    weights = normalize_weights(raw, eps)
    return weights, {a: float(v) for a, v in zip(combo, raw)}


def classwise_recall_weights(combo, model_probs, y_true, num_classes: int, eps: float = 1e-8):
    """
    Peso per classe = recall del modello su quella classe nel validation set.
    Esempio: se DINOv2 Base riconosce bene la classe j, le sue probabilità per j pesano di più.
    """
    weights = np.zeros((num_classes, len(combo)), dtype=float)

    for model_idx, alias in enumerate(combo):
        pred = np.argmax(model_probs[alias], axis=1)
        for class_idx in range(num_classes):
            mask = y_true == class_idx
            if mask.any():
                weights[class_idx, model_idx] = np.mean(pred[mask] == class_idx)
            else:
                weights[class_idx, model_idx] = 0.0

    weights = np.maximum(weights, eps)
    weights = weights / weights.sum(axis=1, keepdims=True)
    return weights


def fuse_probabilities_for_combo(method, combo, model_probs, y_true, num_classes, single_df, config):
    probs_stack = np.stack([model_probs[a] for a in combo], axis=0)

    if method == "soft_voting_uniform":
        fused = probs_stack.mean(axis=0)
        extra = {}

    elif method == "weighted_soft_voting_global":
        metric = config.get("SCREENING_WEIGHT_SOURCE_METRIC", "accuracy")
        weights, raw_scores = global_metric_weights(
            combo,
            single_df,
            metric=metric,
            eps=float(config.get("SCREENING_EPSILON", 1e-8)),
        )
        fused = np.tensordot(weights, probs_stack, axes=(0, 0))
        extra = {
            "global_weights_json": json.dumps({a: float(w) for a, w in zip(combo, weights)}, ensure_ascii=False),
            "raw_global_validation_metric_json": json.dumps(raw_scores, ensure_ascii=False),
            "weight_source_metric": metric,
        }

    elif method == "weighted_soft_voting_per_class":
        class_weights = classwise_recall_weights(
            combo,
            model_probs,
            y_true,
            num_classes,
            eps=float(config.get("SCREENING_EPSILON", 1e-8)),
        )
        # Per ogni classe j, combina la colonna j usando pesi specifici della classe.
        fused = np.zeros_like(model_probs[combo[0]])
        for class_idx in range(num_classes):
            for model_idx, alias in enumerate(combo):
                fused[:, class_idx] += class_weights[class_idx, model_idx] * model_probs[alias][:, class_idx]
        extra = {
            "classwise_weighting": "validation_recall_per_class",
        }

    else:
        raise ValueError(f"Metodo screening non riconosciuto: {method}")

    fused = np.maximum(fused, 1e-12)
    fused = fused / fused.sum(axis=1, keepdims=True)
    return fused, extra



def build_global_single_reference(single_df: pd.DataFrame) -> dict:
    """
    Estrae i migliori valori globali tra tutti i modelli singoli.

    Serve per evitare una valutazione troppo permissiva: un ensemble deve essere
    competitivo anche rispetto al miglior modello singolo globale, non solo rispetto
    ai modelli contenuti nella sua combinazione.
    """
    refs = {}
    higher_is_better = [
        "accuracy",
        "balanced_accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "top3_acc",
        "top5_acc",
        "mean_reciprocal_rank",
    ]
    lower_is_better = [
        "log_loss",
        "expected_calibration_error",
        "mean_true_class_rank",
    ]

    for metric in higher_is_better:
        if metric in single_df.columns:
            values = single_df[metric].astype(float)
            if values.notna().sum() == 0:
                continue
            idx = values.idxmax()
            refs[metric] = {
                "alias": str(single_df.loc[idx, "alias"]),
                "value": float(single_df.loc[idx, metric]),
                "direction": "higher",
            }

    for metric in lower_is_better:
        if metric in single_df.columns:
            values = single_df[metric].astype(float)
            if values.notna().sum() == 0:
                continue
            idx = values.idxmin()
            refs[metric] = {
                "alias": str(single_df.loc[idx, "alias"]),
                "value": float(single_df.loc[idx, metric]),
                "direction": "lower",
            }

    return refs


def enrich_ensemble_row_vs_best_single(row, combo, single_df: pd.DataFrame, config: dict):
    """
    Aggiunge due famiglie di confronti:

    1. confronto locale: ensemble vs miglior modello singolo dentro la stessa combinazione;
    2. confronto globale: ensemble vs miglior modello singolo tra tutti i candidati.

    Il confronto globale è importante perché una combinazione di modelli deboli può
    migliorare rispetto ai suoi componenti, ma restare comunque poco utile rispetto
    al miglior modello singolo complessivo.
    """
    single_by_alias = single_df.set_index("alias")
    combo_single = single_by_alias.loc[list(combo)]
    global_refs = build_global_single_reference(single_df)

    higher_is_better = [
        "accuracy",
        "balanced_accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "top3_acc",
        "top5_acc",
        "mean_reciprocal_rank",
    ]
    lower_is_better = [
        "log_loss",
        "expected_calibration_error",
        "mean_true_class_rank",
    ]

    improved = 0
    worsened = 0
    improved_global = 0
    worsened_global = 0

    # ------------------------------------------------------------
    # 1) Confronto locale: ensemble vs best single nella combo.
    # ------------------------------------------------------------
    for metric in higher_is_better:
        if metric not in row or metric not in combo_single.columns:
            continue
        values = combo_single[metric].astype(float)
        if values.notna().sum() == 0:
            continue
        best_alias = values.idxmax()
        best_val = float(combo_single.loc[best_alias, metric])
        delta = float(row[metric]) - best_val
        row[f"best_single_model_by_{metric}"] = best_alias
        row[f"best_single_{metric}"] = best_val
        row[f"delta_{metric}_vs_best_single"] = delta
        row[f"delta_{metric}_pp_vs_best_single"] = delta * 100.0
        row[f"beats_best_single_{metric}"] = bool(delta > 1e-12)
        if delta > 1e-12:
            improved += 1
        elif delta < -1e-12:
            worsened += 1

    for metric in lower_is_better:
        if metric not in row or metric not in combo_single.columns:
            continue
        values = combo_single[metric].astype(float)
        if values.notna().sum() == 0:
            continue
        best_alias = values.idxmin()
        best_val = float(combo_single.loc[best_alias, metric])
        delta = float(row[metric]) - best_val
        row[f"best_single_model_by_{metric}"] = best_alias
        row[f"best_single_{metric}"] = best_val
        row[f"delta_{metric}_vs_best_single"] = delta
        row[f"beats_best_single_{metric}"] = bool(delta < -1e-12)
        if delta < -1e-12:
            improved += 1
        elif delta > 1e-12:
            worsened += 1

    row["num_metrics_improved_vs_best_single"] = int(improved)
    row["num_metrics_worsened_vs_best_single"] = int(worsened)

    # ------------------------------------------------------------
    # 2) Confronto globale: ensemble vs best single globale.
    # ------------------------------------------------------------
    for metric, ref in global_refs.items():
        if metric not in row:
            continue
        delta = float(row[metric]) - float(ref["value"])
        row[f"best_global_single_model_by_{metric}"] = ref["alias"]
        row[f"best_global_single_{metric}"] = float(ref["value"])
        row[f"delta_{metric}_vs_best_global_single"] = delta

        if ref["direction"] == "higher":
            row[f"delta_{metric}_pp_vs_best_global_single"] = delta * 100.0
            row[f"beats_global_best_single_{metric}"] = bool(delta > 1e-12)
            if delta > 1e-12:
                improved_global += 1
            elif delta < -1e-12:
                worsened_global += 1
        else:
            row[f"beats_global_best_single_{metric}"] = bool(delta < -1e-12)
            if delta < -1e-12:
                improved_global += 1
            elif delta > 1e-12:
                worsened_global += 1

    row["num_metrics_improved_vs_global_best_single"] = int(improved_global)
    row["num_metrics_worsened_vs_global_best_single"] = int(worsened_global)

    # Score locale: misura quanto l'ensemble aggiunge rispetto ai modelli nella combo.
    acc_delta = row.get("delta_accuracy_pp_vs_best_single", 0.0)
    f1_delta = row.get("delta_f1_macro_pp_vs_best_single", 0.0)
    bal_delta = row.get("delta_balanced_accuracy_pp_vs_best_single", 0.0)
    log_delta = row.get("delta_log_loss_vs_best_single", 0.0)
    ece_delta = row.get("delta_expected_calibration_error_vs_best_single", 0.0)

    row["ensemble_recommendation_score"] = float(
        0.45 * acc_delta
        + 0.35 * f1_delta
        + 0.20 * bal_delta
        - 0.50 * max(log_delta, 0.0)
        - 10.0 * max(ece_delta, 0.0)
    )

    # Score globale: misura se l'ensemble è davvero competitivo rispetto al miglior modello singolo globale.
    acc_global_delta = row.get("delta_accuracy_vs_best_global_single", 0.0)
    f1_global_delta = row.get("delta_f1_macro_vs_best_global_single", 0.0)
    bal_global_delta = row.get("delta_balanced_accuracy_vs_best_global_single", 0.0)
    log_global_delta = row.get("delta_log_loss_vs_best_global_single", 0.0)
    ece_global_delta = row.get("delta_expected_calibration_error_vs_best_global_single", 0.0)

    row["global_competitiveness_score"] = float(
        100.0 * (0.50 * acc_global_delta + 0.30 * f1_global_delta + 0.20 * bal_global_delta)
        - 0.50 * max(log_global_delta, 0.0)
        - 10.0 * max(ece_global_delta, 0.0)
    )

    very_acc_delta = float(config.get("SCREENING_VERY_PROMISING_MIN_ACC_DELTA_GLOBAL", 0.003))
    max_acc_drop = float(config.get("SCREENING_PROMISING_MAX_ACC_DROP_GLOBAL", 0.005))
    max_f1_drop = float(config.get("SCREENING_PROMISING_MAX_F1_DROP_GLOBAL", 0.005))
    max_bal_drop = float(config.get("SCREENING_PROMISING_MAX_BAL_ACC_DROP_GLOBAL", 0.007))

    improves_combo = (
        row.get("delta_accuracy_vs_best_single", 0.0) > 0
        and row.get("delta_f1_macro_vs_best_single", 0.0) >= -1e-12
    )
    globally_very_competitive = (
        acc_global_delta >= very_acc_delta
        and f1_global_delta >= -max_f1_drop
        and bal_global_delta >= -max_bal_drop
    )
    globally_competitive = (
        acc_global_delta >= -max_acc_drop
        and f1_global_delta >= -max_f1_drop
        and bal_global_delta >= -max_bal_drop
    )

    if improves_combo and globally_very_competitive:
        row["ensemble_promising_label"] = "very_promising"
    elif improves_combo and globally_competitive:
        row["ensemble_promising_label"] = "promising"
    else:
        row["ensemble_promising_label"] = "not_promising"

    row["globally_competitive_vs_best_single"] = bool(globally_competitive)
    row["improves_over_best_single_in_combo"] = bool(improves_combo)
    return row


def build_selected_for_test_table(ranked_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Produce una tabella corta con le combinazioni da provare sul test set.

    Regola: prima prende gli ensemble very_promising/promising secondo il confronto globale,
    poi limita il numero di righe. Se nessun ensemble supera i criteri, conserva comunque
    il miglior ranking overall come fallback esplicito.
    """
    top_n = int(config.get("SCREENING_SELECTED_FOR_TEST_TOP_N", 8))
    selected = ranked_df[
        ranked_df["ensemble_promising_label"].isin(["very_promising", "promising"])
    ].copy()

    if selected.empty:
        selected = ranked_df.head(min(top_n, len(ranked_df))).copy()
        selected["selection_reason"] = "fallback_top_ranked_no_promising_candidate"
    else:
        selected = selected.head(top_n).copy()
        selected["selection_reason"] = "promising_validation_candidate"

    columns_first = [
        "rank",
        "selection_reason",
        "ensemble_promising_label",
        "ensemble_method",
        "combo_size",
        "combo_models",
        "accuracy",
        "f1_macro",
        "balanced_accuracy",
        "log_loss",
        "expected_calibration_error",
        "delta_accuracy_pp_vs_best_global_single",
        "delta_f1_macro_pp_vs_best_global_single",
        "delta_balanced_accuracy_pp_vs_best_global_single",
        "best_global_single_model_by_accuracy",
        "best_global_single_accuracy",
        "delta_accuracy_pp_vs_best_single",
        "best_single_model_by_accuracy",
        "best_single_accuracy",
        "num_metrics_improved_vs_global_best_single",
        "num_metrics_worsened_vs_global_best_single",
    ]
    existing_first = [col for col in columns_first if col in selected.columns]
    remaining = [col for col in selected.columns if col not in existing_first]
    return selected[existing_first + remaining]



def add_validation_screening_alias_columns(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Aggiunge colonne esplicite per evitare ambiguità tra validation/OOF e test finale.

    Le colonne originali (`accuracy`, `f1_macro`, `balanced_accuracy`, ecc.) restano presenti
    per compatibilità interna, ma le colonne `validation_screening_*` chiariscono che questi
    valori derivano dalla fase di screening su validation.
    """
    df = df.copy()
    sampling_mode = str(config.get("SAMPLING_MODE", "")).lower()
    scope = "cv_oof_validation" if sampling_mode == "subsampling" else "holdout_validation"
    df["validation_screening_scope"] = scope

    rename_like = {
        "accuracy": "validation_screening_accuracy",
        "f1_macro": "validation_screening_f1_macro",
        "balanced_accuracy": "validation_screening_balanced_accuracy",
        "log_loss": "validation_screening_log_loss",
        "expected_calibration_error": "validation_screening_expected_calibration_error",
        "delta_accuracy_pp_vs_best_global_single": "delta_validation_screening_accuracy_pp_vs_best_global_single",
        "delta_f1_macro_pp_vs_best_global_single": "delta_validation_screening_f1_macro_pp_vs_best_global_single",
        "delta_balanced_accuracy_pp_vs_best_global_single": "delta_validation_screening_balanced_accuracy_pp_vs_best_global_single",
        "best_global_single_accuracy": "best_global_single_validation_screening_accuracy",
    }

    for old_col, new_col in rename_like.items():
        if old_col in df.columns and new_col not in df.columns:
            df[new_col] = df[old_col]

    return df


def print_validation_screening_note(config: dict):
    """Stampa una nota breve per non confondere validation/OOF con test finale."""
    sampling_mode = str(config.get("SAMPLING_MODE", "")).lower()
    print("\nNota metriche screening:")
    if sampling_mode == "subsampling":
        print("- SAMPLING_MODE='subsampling': queste metriche sono calcolate sulle OOF probabilities della cross-validation.")
        print("- Non sono direttamente confrontabili con la test accuracy del modello final_refit.")
        print("- Il delta rispetto al best single model usa il riferimento CV/OOF dei singoli modelli.")
    else:
        print("- SAMPLING_MODE='upsampling': queste metriche sono calcolate sul validation set hold-out non augmentato.")
        print("- Il delta rispetto al best single model è un delta di validation screening, non di test finale.")

def run_validation_probability_screening_from_files(config: dict):
    """
    Crea tabelle condensate simili ai file validation_exhaustive_* originali,
    ma in meno file e usando direttamente i CSV di probabilità validation.
    """
    output_root = Path(config["OUTPUT_DIR"]) / "validation_probability_screening"
    output_root.mkdir(parents=True, exist_ok=True)

    bundle = load_validation_probability_files(config)
    y_true = bundle["y_true"]
    model_probs = bundle["model_probs"]
    num_classes = bundle["num_classes"]

    single_df = compute_single_model_screening(bundle)
    single_df = add_validation_screening_alias_columns(single_df, config)
    single_path = output_root / "validation_model_screening_single_models.csv"
    single_df.to_csv(single_path, index=False, encoding="utf-8-sig")

    aliases = list(model_probs.keys())
    combos = build_screening_candidate_combinations(config, aliases)
    methods = config.get("SCREENING_PROBABILITY_METHODS", ["soft_voting_uniform"])

    rows = []
    for combo_id, combo in enumerate(combos, start=1):
        for method in methods:
            fused, extra = fuse_probabilities_for_combo(
                method=method,
                combo=combo,
                model_probs=model_probs,
                y_true=y_true,
                num_classes=num_classes,
                single_df=single_df,
                config=config,
            )

            row = {
                "candidate_type": "probability_ensemble",
                "ensemble_method": method,
                "combo_size": len(combo),
                "combo_models": "|".join(combo),
                "num_models": len(combo),
                "combo_id": combo_id,
            }
            row.update(extra)
            row.update(compute_screening_metrics(y_true, fused, num_classes))
            row = enrich_ensemble_row_vs_best_single(row, combo, single_df, config)
            rows.append(row)

    ranked_df = pd.DataFrame(rows)
    selection_metric = config.get("SCREENING_SELECTION_METRIC", "accuracy")
    ascending = selection_metric in {"log_loss", "expected_calibration_error", "mean_true_class_rank"}
    ranked_df = ranked_df.sort_values(
        by=[selection_metric, "f1_macro", "balanced_accuracy"],
        ascending=[ascending, False, False],
    ).reset_index(drop=True)
    ranked_df.insert(0, "rank", range(1, len(ranked_df) + 1))
    ranked_df = add_validation_screening_alias_columns(ranked_df, config)

    ranked_path = output_root / "validation_probability_ensemble_screening_ranked.csv"
    ranked_df.to_csv(ranked_path, index=False, encoding="utf-8-sig")

    selected_for_test_df = build_selected_for_test_table(ranked_df, config)
    selected_for_test_path = output_root / "validation_probability_selected_for_test.csv"
    selected_for_test_df.to_csv(selected_for_test_path, index=False, encoding="utf-8-sig")

    # File condensato: best overall, best per metodo, best per size e top promising.
    summary_parts = []

    best_overall = ranked_df.head(1).copy()
    best_overall.insert(0, "summary_type", "best_overall")
    summary_parts.append(best_overall)

    best_per_method = (
        ranked_df.sort_values("rank")
        .groupby("ensemble_method", as_index=False)
        .head(1)
        .copy()
    )
    best_per_method.insert(0, "summary_type", "best_per_method")
    summary_parts.append(best_per_method)

    best_per_size = (
        ranked_df.sort_values("rank")
        .groupby("combo_size", as_index=False)
        .head(1)
        .copy()
    )
    best_per_size.insert(0, "summary_type", "best_per_size")
    summary_parts.append(best_per_size)

    top_n = int(config.get("SCREENING_TOP_N_SUMMARY", 30))
    promising = (
        ranked_df[
            (ranked_df["ensemble_promising_label"].isin(["very_promising", "promising"]))
        ]
        .head(top_n)
        .copy()
    )
    if len(promising) > 0:
        promising.insert(0, "summary_type", "top_promising")
        summary_parts.append(promising)

    summary_df = pd.concat(summary_parts, ignore_index=True)
    summary_path = output_root / "validation_probability_ensemble_screening_summary.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    summary_json = {
        "screening_mode": "validation_probabilities_from_files",
        "sampling_mode": config.get("SAMPLING_MODE"),
        "num_models": len(aliases),
        "num_classes": int(num_classes),
        "num_validation_samples": int(len(y_true)),
        "num_combinations": int(len(combos)),
        "num_evaluations": int(len(ranked_df)),
        "methods": methods,
        "selection_metric": selection_metric,
        "alignment_key": bundle["alignment_key"],
        "metric_note": (
            "subsampling uses cross-validation OOF probabilities; validation screening metrics are not directly comparable "
            "with final_refit test accuracy."
            if str(config.get("SAMPLING_MODE", "")).lower() == "subsampling"
            else "upsampling uses the hold-out validation set; screening metrics are not final test metrics."
        ),
        "best_single_model_by_accuracy": single_df.iloc[0]["alias"],
        "best_single_accuracy": float(single_df.iloc[0]["accuracy"]),
        "best_ensemble": ranked_df.iloc[0].to_dict() if len(ranked_df) else None,
        "outputs": {
            "single_models": str(single_path),
            "ranked_ensembles": str(ranked_path),
            "summary": str(summary_path),
            "selected_for_test": str(selected_for_test_path),
        },
        "promising_criteria": {
            "very_promising_min_acc_delta_global": float(config.get("SCREENING_VERY_PROMISING_MIN_ACC_DELTA_GLOBAL", 0.003)),
            "promising_max_acc_drop_global": float(config.get("SCREENING_PROMISING_MAX_ACC_DROP_GLOBAL", 0.005)),
            "promising_max_f1_drop_global": float(config.get("SCREENING_PROMISING_MAX_F1_DROP_GLOBAL", 0.005)),
            "promising_max_bal_acc_drop_global": float(config.get("SCREENING_PROMISING_MAX_BAL_ACC_DROP_GLOBAL", 0.007)),
            "selected_for_test_top_n": int(config.get("SCREENING_SELECTED_FOR_TEST_TOP_N", 8)),
        },
    }

    json_path = output_root / "validation_probability_screening_summary.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(summary_json), f, ensure_ascii=False, indent=2)

    print("\nScreening validation completato.")
    print_validation_screening_note(config)
    print(f"Single models: {single_path}")
    print(f"Ranking ensemble: {ranked_path}")
    print(f"Summary condensato: {summary_path}")
    print(f"Selected for test: {selected_for_test_path}")
    print(f"JSON summary: {json_path}")

    return {
        "single_models_path": str(single_path),
        "ranked_ensembles_path": str(ranked_path),
        "summary_path": str(summary_path),
        "selected_for_test_path": str(selected_for_test_path),
        "summary_json_path": str(json_path),
        # Mantengo i DataFrame nel dizionario per eventuale ispezione in notebook,
        # ma l'ultima cella non li stampa più automaticamente.
        "single_models": single_df,
        "ranked_ensembles": ranked_df,
        "selected_for_test": selected_for_test_df,
        "summary": summary_df,
    }


In [ ]:
# ============================================================
# 6. VALIDATION COMBINAZIONI + TEST FINALE
# ============================================================

def model_cfg_by_alias(config: dict):
    return {run_cfg["alias"]: run_cfg for run_cfg in config["BASE_MODELS"]}


def build_candidate_combinations(config: dict):
    """Genera o legge le combinazioni di modelli da validare."""
    aliases = [run_cfg["alias"] for run_cfg in config["BASE_MODELS"]]

    manual = config.get("CANDIDATE_COMBINATIONS")
    if manual:
        combos = [tuple(combo) for combo in manual]
    else:
        min_size = int(config["MIN_COMBO_SIZE"])
        max_size = min(int(config["MAX_COMBO_SIZE"]), len(aliases))
        combos = []
        for size in range(min_size, max_size + 1):
            combos.extend(itertools.combinations(aliases, size))

    unknown = sorted({alias for combo in combos for alias in combo if alias not in aliases})
    if unknown:
        raise ValueError(f"Alias non presenti in BASE_MODELS: {unknown}")

    return combos


def combo_name(combo):
    return "__".join(combo)



def make_json_safe(obj):
    """Converte numpy/pandas objects in tipi serializzabili da json.dump."""
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj) if not isinstance(obj, (list, tuple, dict, np.ndarray)) else False:
        return None
    return obj


def metric_value(metrics: dict, metric_name: str):
    value = metrics.get(metric_name)
    if value is None:
        return -np.inf
    return float(value)


def extract_outputs_for_all_base_models(split_dfs: dict, output_dir: Path, config: dict, num_classes: int):
    """
    Estrae/cachea una sola volta probabilità ed embedding per ogni modello e split.
    Poi tutte le combinazioni riusano questi array, evitando forward pass ripetuti.
    """
    outputs = {}

    for run_cfg in config["BASE_MODELS"]:
        alias = run_cfg["alias"]
        outputs[alias] = {}
        print("\n" + "=" * 70)
        print("Modello base:", alias)

        for split_name, df_split in split_dfs.items():
            outputs[alias][split_name] = extract_or_load_model_outputs(
                run_cfg=run_cfg,
                split_name=split_name,
                df_split=df_split,
                output_dir=output_dir,
                config=config,
                num_classes=num_classes,
            )

    # Controllo forte: per ogni split, tutti i modelli devono produrre label nello stesso ordine.
    for split_name in split_dfs:
        reference = None
        for alias in outputs:
            labels = outputs[alias][split_name]["labels"]
            if reference is None:
                reference = labels
            elif not np.array_equal(reference, labels):
                raise ValueError(f"Label non allineate nello split {split_name} per il modello {alias}.")

    return outputs


def probability_stack(outputs: dict, combo: tuple, split_name: str):
    """Restituisce stack [M, N, C] delle probabilità dei modelli della combinazione."""
    return np.stack([outputs[alias][split_name]["probabilities"] for alias in combo], axis=0)


def combine_probabilities(prob_stack: np.ndarray, weights=None, class_weights=None):
    """
    Combina probabilità finali dei modelli base.
    - weights: vettore [M] per weighted soft voting globale;
    - class_weights: matrice [M, C] per pesi diversi per classe.
    """
    if class_weights is not None:
        class_weights = np.asarray(class_weights, dtype=np.float64)
        combined = np.sum(prob_stack * class_weights[:, None, :], axis=0)
    else:
        if weights is None:
            weights = np.ones(prob_stack.shape[0], dtype=np.float64) / prob_stack.shape[0]
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
        combined = np.tensordot(weights, prob_stack, axes=(0, 0))

    combined = normalize_probabilities(combined)
    return combined.astype(np.float32)


def generate_weight_grid(n_models: int, step: float = 0.10):
    """Genera pesi non negativi che sommano a 1 usando una griglia semplice."""
    total = int(round(1.0 / step))
    if total <= 0:
        raise ValueError("WEIGHT_GRID_STEP non valido.")

    def compositions(total_value, parts):
        if parts == 1:
            yield [total_value]
        else:
            for value in range(total_value + 1):
                for rest in compositions(total_value - value, parts - 1):
                    yield [value] + rest

    for integers in compositions(total, n_models):
        weights = np.asarray(integers, dtype=np.float64) / total
        if weights.sum() > 0:
            yield weights


def find_best_global_weights(y_val, prob_stack_val, num_classes: int, config: dict):
    """Cerca i pesi globali migliori in validation."""
    selection_metric = config["SELECTION_METRIC"]
    best = None

    for weights in generate_weight_grid(prob_stack_val.shape[0], step=float(config["WEIGHT_GRID_STEP"])):
        prob = combine_probabilities(prob_stack_val, weights=weights)
        metrics, _ = compute_metrics(y_val, prob, num_classes)
        score = metric_value(metrics, selection_metric)
        logloss = metrics.get("log_loss") if metrics.get("log_loss") is not None else np.inf

        candidate = {
            "weights": weights,
            "metrics": metrics,
            "score": score,
            "log_loss": logloss,
        }

        if best is None:
            best = candidate
        elif (score > best["score"]) or (score == best["score"] and logloss < best["log_loss"]):
            best = candidate

    return best["weights"], best["metrics"]


def compute_per_class_weights(y_val, prob_stack_val, num_classes: int, smoothing: float = 1e-3):
    """
    Costruisce pesi per classe in base alla recall validation dei singoli modelli.
    Per ogni classe c: peso modello m ∝ recall_m,c + smoothing.
    """
    y_val = np.asarray(y_val)
    preds = np.argmax(prob_stack_val, axis=2)  # [M, N]
    class_weights = np.zeros((prob_stack_val.shape[0], num_classes), dtype=np.float64)

    for c in range(num_classes):
        class_mask = y_val == c
        for m in range(prob_stack_val.shape[0]):
            if class_mask.any():
                recall_c = np.mean(preds[m, class_mask] == c)
            else:
                recall_c = 0.0
            class_weights[m, c] = recall_c + smoothing

        total = class_weights[:, c].sum()
        if total <= 0:
            class_weights[:, c] = 1.0 / prob_stack_val.shape[0]
        else:
            class_weights[:, c] /= total

    return class_weights


def run_probability_ensemble_validation(outputs: dict, combos: list, y_val: np.ndarray, num_classes: int, output_dir: Path, config: dict):
    """Valida le combinazioni usando solo le probabilità finali dei modelli base."""
    rows = []
    details = {}

    if not config.get("RUN_PROBABILITY_ENSEMBLES", True):
        return pd.DataFrame(), details

    for combo in combos:
        prob_stack_val = probability_stack(outputs, combo, "val")
        c_name = combo_name(combo)
        details[c_name] = {}

        if "soft_voting_uniform" in config["PROBABILITY_METHODS"]:
            prob = combine_probabilities(prob_stack_val)
            metrics, _ = compute_metrics(y_val, prob, num_classes)
            rows.append({
                "method": "soft_voting_uniform",
                "combo": c_name,
                "models": list(combo),
                "weights": json.dumps([1.0 / len(combo)] * len(combo)),
                "class_weights": None,
                **{f"val_{k}": v for k, v in metrics.items()},
            })

        if "weighted_soft_voting_global" in config["PROBABILITY_METHODS"]:
            weights, metrics = find_best_global_weights(y_val, prob_stack_val, num_classes, config)
            details[c_name]["weighted_soft_voting_global_weights"] = weights
            rows.append({
                "method": "weighted_soft_voting_global",
                "combo": c_name,
                "models": list(combo),
                "weights": json.dumps(weights.tolist()),
                "class_weights": None,
                **{f"val_{k}": v for k, v in metrics.items()},
            })

        if "weighted_soft_voting_per_class" in config["PROBABILITY_METHODS"]:
            class_weights = compute_per_class_weights(y_val, prob_stack_val, num_classes)
            prob = combine_probabilities(prob_stack_val, class_weights=class_weights)
            metrics, _ = compute_metrics(y_val, prob, num_classes)
            details[c_name]["weighted_soft_voting_per_class_weights"] = class_weights
            rows.append({
                "method": "weighted_soft_voting_per_class",
                "combo": c_name,
                "models": list(combo),
                "weights": None,
                "class_weights": json.dumps(class_weights.tolist()),
                **{f"val_{k}": v for k, v in metrics.items()},
            })

    ranking = pd.DataFrame(rows)
    if not ranking.empty:
        ranking = ranking.sort_values(
            by=[f"val_{config['SELECTION_METRIC']}", "val_log_loss"],
            ascending=[False, True],
        ).reset_index(drop=True)
        ranking.to_csv(output_dir / "validation_probability_ensemble_ranking.csv", index=False, encoding="utf-8-sig")

    return ranking, details


def concatenate_features(outputs: dict, combo: tuple, split_name: str):
    """Concatena embedding dei modelli della combinazione su uno split."""
    features = [outputs[alias][split_name]["features"] for alias in combo]
    labels = outputs[combo[0]][split_name]["labels"]
    X = np.concatenate(features, axis=1).astype(np.float32)
    y = labels.astype(np.int64)
    return X, y


def fit_late_fusion_for_combo(outputs: dict, combo: tuple, output_dir: Path, config: dict, num_classes: int):
    """Allena il classificatore finale sulle feature concatenate della singola combinazione."""
    combo_dir = output_dir / "late_fusion_models" / combo_name(combo)
    combo_dir.mkdir(parents=True, exist_ok=True)

    X_train, y_train = concatenate_features(outputs, combo, "train")
    X_val, y_val = concatenate_features(outputs, combo, "val")

    classifier_name = config["FUSION_CLASSIFIER"].lower()

    if classifier_name == "mlp":
        classifier, classifier_info = fit_mlp_classifier(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            config=config,
            output_dir=combo_dir,
            num_classes=num_classes,
        )
    elif classifier_name == "logistic_regression":
        classifier, classifier_info = fit_logistic_regression_classifier(
            X_train=X_train,
            y_train=y_train,
            config=config,
            output_dir=combo_dir,
        )
    else:
        raise ValueError("FUSION_CLASSIFIER deve essere 'mlp' oppure 'logistic_regression'.")

    val_prob = predict_classifier_probabilities(classifier, X_val, y_val, config)
    val_metrics, _ = compute_metrics(y_val, val_prob, num_classes)

    return classifier, classifier_info, val_metrics


def run_late_fusion_validation(outputs: dict, combos: list, num_classes: int, output_dir: Path, config: dict):
    """Valida tutte le combinazioni con late fusion sugli embedding."""
    rows = []
    classifiers = {}

    if not config.get("RUN_LATE_FUSION_EMBEDDINGS", True):
        return pd.DataFrame(), classifiers

    for combo in combos:
        c_name = combo_name(combo)
        print("\n" + "=" * 70)
        print("Late fusion embedding combo:", c_name)

        classifier, classifier_info, val_metrics = fit_late_fusion_for_combo(
            outputs=outputs,
            combo=combo,
            output_dir=output_dir,
            config=config,
            num_classes=num_classes,
        )

        classifiers[c_name] = {
            "classifier": classifier,
            "classifier_info": classifier_info,
        }

        X_train, _ = concatenate_features(outputs, combo, "train")
        rows.append({
            "method": f"late_fusion_embeddings_{config['FUSION_CLASSIFIER']}",
            "combo": c_name,
            "models": list(combo),
            "input_dim": int(X_train.shape[1]),
            "classifier_info": json.dumps(classifier_info),
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

    ranking = pd.DataFrame(rows)
    if not ranking.empty:
        ranking = ranking.sort_values(
            by=[f"val_{config['SELECTION_METRIC']}", "val_log_loss"],
            ascending=[False, True],
        ).reset_index(drop=True)
        ranking.to_csv(output_dir / "validation_late_fusion_embedding_ranking.csv", index=False, encoding="utf-8-sig")

    return ranking, classifiers


def select_rows_for_final_test(validation_ranking: pd.DataFrame, config: dict):
    """Seleziona le migliori combinazioni da portare su test finale."""
    if validation_ranking.empty:
        return validation_ranking

    selection_metric = f"val_{config['SELECTION_METRIC']}"
    top_n = int(config.get("TEST_TOP_N_PER_METHOD", 1))

    selected = (
        validation_ranking
        .sort_values(by=[selection_metric, "val_log_loss"], ascending=[False, True])
        .groupby("method", group_keys=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return selected


def evaluate_probability_method_on_test(row: pd.Series, outputs: dict, y_test: np.ndarray, num_classes: int, output_dir: Path):
    method = row["method"]
    combo = tuple(row["models"] if isinstance(row["models"], list) else json.loads(row["models"].replace("'", '"')))
    c_name = combo_name(combo)
    prob_stack_test = probability_stack(outputs, combo, "test")

    if method == "soft_voting_uniform":
        prob = combine_probabilities(prob_stack_test)
    elif method == "weighted_soft_voting_global":
        weights = np.asarray(json.loads(row["weights"]), dtype=np.float64)
        prob = combine_probabilities(prob_stack_test, weights=weights)
    elif method == "weighted_soft_voting_per_class":
        class_weights = np.asarray(json.loads(row["class_weights"]), dtype=np.float64)
        prob = combine_probabilities(prob_stack_test, class_weights=class_weights)
    else:
        raise ValueError(f"Metodo probabilistico non supportato: {method}")

    metrics, pred = compute_metrics(y_test, prob, num_classes)

    pred_df = pd.DataFrame({
        "true_label_idx": y_test,
        "pred_label_idx": pred,
        "is_correct": y_test == pred,
    })
    for class_idx in range(num_classes):
        pred_df[f"prob_{class_idx}"] = prob[:, class_idx]

    pred_path = output_dir / f"test_predictions__{method}__{c_name}.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    return metrics, str(pred_path)


def fit_final_late_fusion_classifier_for_test(outputs: dict, combo: tuple, output_dir: Path, config: dict, num_classes: int):
    """
    Per il subsampling, dopo la selezione su fold validation, ri-addestra il classificatore
    di late fusion sugli embedding del CV pool completo prodotti dai checkpoint final_refit.
    Questo non riaddestra le reti base: addestra solo il piccolo classificatore finale.
    """
    combo_dir = output_dir / "late_fusion_models_final_test" / combo_name(combo)
    combo_dir.mkdir(parents=True, exist_ok=True)

    X_final, y_final = concatenate_features(outputs, combo, "final_train")
    classifier_name = config["FUSION_CLASSIFIER"].lower()

    if classifier_name == "mlp":
        # Split interno del CV pool solo per early stopping del classificatore finale.
        # Il test set resta completamente separato.
        val_size = float(config.get("FINAL_FUSION_VAL_SIZE", 0.1111))
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=int(config["SEED"]))
        train_idx, val_idx = next(splitter.split(X_final, y_final))

        print(
            "\n[REFIT FINALE LATE FUSION]"
            "\nLe metriche val_acc/val_f1 stampate nelle prossime epoche sono calcolate"
            "\nsu uno split interno del final_train usato solo per early stopping."
            "\nNON sono metriche sul test set."
        )

        classifier, classifier_info = fit_mlp_classifier(
            X_train=X_final[train_idx],
            y_train=y_final[train_idx],
            X_val=X_final[val_idx],
            y_val=y_final[val_idx],
            config=config,
            output_dir=combo_dir,
            num_classes=num_classes,
        )
        classifier_info["final_test_refit"] = True
        classifier_info["final_train_samples"] = int(len(y_final))
        classifier_info["internal_val_size"] = val_size

    elif classifier_name == "logistic_regression":
        classifier, classifier_info = fit_logistic_regression_classifier(
            X_train=X_final,
            y_train=y_final,
            config=config,
            output_dir=combo_dir,
        )
        classifier_info["final_test_refit"] = True
        classifier_info["final_train_samples"] = int(len(y_final))

    else:
        raise ValueError("FUSION_CLASSIFIER deve essere 'mlp' oppure 'logistic_regression'.")

    return classifier, classifier_info


def evaluate_late_fusion_on_test(row: pd.Series, outputs: dict, classifiers: dict, y_test: np.ndarray, num_classes: int, output_dir: Path, config: dict):
    combo = tuple(row["models"] if isinstance(row["models"], list) else json.loads(row["models"].replace("'", '"')))
    c_name = combo_name(combo)

    if (
        config["SAMPLING_MODE"] == "subsampling"
        and config.get("REFIT_FUSION_CLASSIFIER_FOR_SUBSAMPLING_TEST", True)
        and "final_train" in outputs[combo[0]]
    ):
        # Validation: ranking su fold checkpoint.
        # Test finale: ri-addestra solo la testa/fusion classifier su CV pool con final_refit.
        classifier, classifier_info = fit_final_late_fusion_classifier_for_test(
            outputs=outputs,
            combo=combo,
            output_dir=output_dir,
            config=config,
            num_classes=num_classes,
        )
        classifiers[c_name + "__final_test_refit"] = {
            "classifier": classifier,
            "classifier_info": classifier_info,
        }
    else:
        classifier = classifiers[c_name]["classifier"]

    X_test, y_test_from_features = concatenate_features(outputs, combo, "test")
    if not np.array_equal(y_test, y_test_from_features):
        raise ValueError(f"Label test non allineate per late fusion {c_name}.")

    prob = predict_classifier_probabilities(classifier, X_test, y_test, config)
    metrics, pred = compute_metrics(y_test, prob, num_classes)

    pred_df = pd.DataFrame({
        "true_label_idx": y_test,
        "pred_label_idx": pred,
        "is_correct": y_test == pred,
    })
    for class_idx in range(num_classes):
        pred_df[f"prob_{class_idx}"] = prob[:, class_idx]

    pred_path = output_dir / f"test_predictions__{row['method']}__{c_name}.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    return metrics, str(pred_path)


def validate_sampling_mode_and_checkpoints(config: dict):
    """
    Controlla coerenza tra SAMPLING_MODE e checkpoint.

    Per subsampling sono supportati due path standard:
    - checkpoint_path = <model>_final_refit.pt;
    - fold_checkpoint_path opzionale = <model>_fold_{fold}.pt.

    Se fold_checkpoint_path manca, viene inferito automaticamente dalla cartella models.
    """
    mode = str(config["SAMPLING_MODE"]).lower()
    if mode not in {"upsampling", "subsampling"}:
        raise ValueError("SAMPLING_MODE deve essere 'upsampling' oppure 'subsampling'.")

    paths = [str(m.get("checkpoint_path", "")) for m in config["BASE_MODELS"]]
    placeholder_paths = [p for p in paths if "YYYYMMDD" in p or "HHMMSS" in p]
    if placeholder_paths:
        print("[ATTENZIONE] Alcuni checkpoint_path contengono ancora placeholder YYYYMMDD_HHMMSS.")
        print("Aggiorna i path reali prima di eseguire il notebook end-to-end.")

    wrong_paths = []
    for p in paths:
        p_low = p.lower()
        if mode == "upsampling" and "/subsampling/" in p_low:
            wrong_paths.append(p)
        if mode == "subsampling" and "/upsampling/" in p_low:
            wrong_paths.append(p)

    if wrong_paths:
        formatted = "\n".join(wrong_paths)
        raise ValueError(
            f"Checkpoint non coerenti con SAMPLING_MODE='{mode}'.\n"
            f"Path problematici:\n{formatted}\n\n"
            "Correzione: usa checkpoint in results/upsampling/... per SAMPLING_MODE='upsampling' "
            "e checkpoint in results/subsampling/... per SAMPLING_MODE='subsampling'."
        )

    if mode == "subsampling":
        print("\n[SUBSAMPLING ADATTIVO]")
        print("Validation combinazioni: checkpoint fold-specific sul fold scelto.")
        print("Test finale: checkpoint final_refit sul test split separato.")
        for run_cfg in config["BASE_MODELS"]:
            fold_p = infer_subsampling_fold_checkpoint_path(run_cfg, config)
            final_p = infer_subsampling_final_refit_checkpoint_path(run_cfg)
            print(f"- {run_cfg['alias']} | fold checkpoint: {fold_p.name} | final refit: {final_p.name}")
            if "YYYYMMDD" not in str(fold_p) and "HHMMSS" not in str(fold_p) and not fold_p.exists():
                print(f"  [ATTENZIONE] fold checkpoint non trovato ora: {fold_p}")
            if "YYYYMMDD" not in str(final_p) and "HHMMSS" not in str(final_p) and not final_p.exists():
                print(f"  [ATTENZIONE] final_refit checkpoint non trovato ora: {final_p}")


def run_ensemble_late_fusion(config: dict):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(config["OUTPUT_DIR"]) / f"ensemble_late_fusion_{config['SAMPLING_MODE']}__{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)

    print("Output dir:", output_dir)
    validate_sampling_mode_and_checkpoints(config)
    prepare_local_dataset(config)

    train_df, val_df, test_df, final_train_df, class_mapping_df, split_info, num_classes = load_late_fusion_metadata(config)

    # Nel subsampling usiamo checkpoint diversi in base allo split:
    # train/val -> checkpoint fold-specific per validazione;
    # final_train/test -> checkpoint final_refit per test finale separato.
    if config["SAMPLING_MODE"] == "subsampling":
        split_dfs = {"train": train_df, "val": val_df, "final_train": final_train_df, "test": test_df}
    else:
        split_dfs = {"train": train_df, "val": val_df, "test": test_df}
    combos = build_candidate_combinations(config)

    print("Combinazioni candidate:")
    for combo in combos:
        print("-", combo_name(combo))

    outputs = extract_outputs_for_all_base_models(
        split_dfs=split_dfs,
        output_dir=output_dir,
        config=config,
        num_classes=num_classes,
    )

    y_val = outputs[config["BASE_MODELS"][0]["alias"]]["val"]["labels"]
    y_test = outputs[config["BASE_MODELS"][0]["alias"]]["test"]["labels"]

    probability_ranking, probability_details = run_probability_ensemble_validation(
        outputs=outputs,
        combos=combos,
        y_val=y_val,
        num_classes=num_classes,
        output_dir=output_dir,
        config=config,
    )

    late_fusion_ranking, late_fusion_classifiers = run_late_fusion_validation(
        outputs=outputs,
        combos=combos,
        num_classes=num_classes,
        output_dir=output_dir,
        config=config,
    )

    validation_frames = []
    if not probability_ranking.empty:
        validation_frames.append(probability_ranking)
    if not late_fusion_ranking.empty:
        validation_frames.append(late_fusion_ranking)

    if validation_frames:
        validation_ranking = pd.concat(validation_frames, ignore_index=True, sort=False)
        validation_ranking = validation_ranking.sort_values(
            by=[f"val_{config['SELECTION_METRIC']}", "val_log_loss"],
            ascending=[False, True],
        ).reset_index(drop=True)
    else:
        raise RuntimeError("Nessun metodo ensemble/late fusion abilitato.")

    validation_ranking.to_csv(output_dir / "validation_all_methods_ranking.csv", index=False, encoding="utf-8-sig")

    selected_for_test = select_rows_for_final_test(validation_ranking, config)
    selected_for_test.to_csv(output_dir / "selected_combinations_for_test.csv", index=False, encoding="utf-8-sig")

    test_rows = []
    for _, row in selected_for_test.iterrows():
        method = row["method"]
        print("\nTEST FINALE:", method, "|", row["combo"])

        if method.startswith("late_fusion_embeddings"):
            test_metrics, pred_path = evaluate_late_fusion_on_test(
                row=row,
                outputs=outputs,
                classifiers=late_fusion_classifiers,
                y_test=y_test,
                num_classes=num_classes,
                output_dir=output_dir,
                config=config,
            )
        else:
            test_metrics, pred_path = evaluate_probability_method_on_test(
                row=row,
                outputs=outputs,
                y_test=y_test,
                num_classes=num_classes,
                output_dir=output_dir,
            )

        test_rows.append({
            "method": method,
            "combo": row["combo"],
            "models": row["models"],
            **{f"val_{k.replace('val_', '')}": row[k] for k in row.index if isinstance(k, str) and k.startswith("val_")},
            **{f"test_{k}": v for k, v in test_metrics.items()},
            "test_predictions_path": pred_path,
        })

    test_results = pd.DataFrame(test_rows)
    test_results = test_results.sort_values(
        by=[f"test_{config['SELECTION_METRIC']}", "test_log_loss"],
        ascending=[False, True],
    ).reset_index(drop=True)
    test_results.to_csv(output_dir / "test_final_results.csv", index=False, encoding="utf-8-sig")

    summary = {
        "sampling_mode": config["SAMPLING_MODE"],
        "dataset_variant": "original",
        "split_info": split_info,
        "num_classes": int(num_classes),
        "base_models": config["BASE_MODELS"],
        "candidate_combinations": [list(combo) for combo in combos],
        "selection_metric": config["SELECTION_METRIC"],
        "files": {
            "output_dir": str(output_dir),
            "validation_all_methods_ranking": str(output_dir / "validation_all_methods_ranking.csv"),
            "selected_combinations_for_test": str(output_dir / "selected_combinations_for_test.csv"),
            "test_final_results": str(output_dir / "test_final_results.csv"),
        },
        "best_validation": validation_ranking.head(10).to_dict(orient="records"),
        "test_results": test_results.to_dict(orient="records"),
    }

    with open(output_dir / "ensemble_late_fusion_summary.json", "w", encoding="utf-8") as f:
        json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

    print("\n===== TOP VALIDATION =====")
    display_cols = ["method", "combo", f"val_{config['SELECTION_METRIC']}", "val_macro_f1", "val_log_loss"]
    print(validation_ranking[display_cols].head(20).to_string(index=False))

    print("\n===== TEST FINALE =====")
    display_cols = ["method", "combo", f"test_{config['SELECTION_METRIC']}", "test_macro_f1", "test_log_loss"]
    print(test_results[display_cols].to_string(index=False))

    print("\nSummary salvato in:", output_dir / "ensemble_late_fusion_summary.json")
    return summary


In [ ]:
# ============================================================
# 7. ESECUZIONE
# ============================================================
outputs = {}

if CONFIG.get("RUN_VALIDATION_PROBABILITY_SCREENING_FROM_FILES", False):
    outputs["validation_probability_screening"] = run_validation_probability_screening_from_files(CONFIG)

if CONFIG.get("RUN_CHECKPOINT_BASED_ENSEMBLE_LATE_FUSION", True):
    outputs["checkpoint_based_ensemble_late_fusion"] = run_ensemble_late_fusion(CONFIG)

print("\nEsecuzione completata.")
print("Blocchi eseguiti:", ", ".join(outputs.keys()) if outputs else "nessuno")

if "validation_probability_screening" in outputs:
    s = outputs["validation_probability_screening"]
    print("\nFile screening validation:")
    print("- Single models:", s["single_models_path"])
    print("- Ranking ensemble:", s["ranked_ensembles_path"])
    print("- Summary:", s["summary_path"])
    print("- Selected for test:", s["selected_for_test_path"])
    print("- JSON:", s["summary_json_path"])

    print("\nTop combinazioni selezionate per il test:")
    print("Nota: le colonne validation_screening_* sono metriche di selezione su validation/OOF, non test finale.")
    display_cols = [
        "rank",
        "ensemble_promising_label",
        "ensemble_method",
        "combo_size",
        "combo_models",
        "validation_screening_accuracy",
        "validation_screening_f1_macro",
        "validation_screening_balanced_accuracy",
        "delta_validation_screening_accuracy_pp_vs_best_global_single",
        "validation_screening_scope",
    ]
    display_cols = [c for c in display_cols if c in s["selected_for_test"].columns]
    display(s["selected_for_test"][display_cols].head(10))


Caricati 17 modelli su 816 campioni validation.
Numero classi: 46. Chiave allineamento: image_path

Screening validation completato.

Nota metriche screening:
- SAMPLING_MODE='upsampling': queste metriche sono calcolate sul validation set hold-out non augmentato.
- Il delta rispetto al best single model è un delta di validation screening, non di test finale.
Single models: /content/drive/MyDrive/Dataset Fargetta_224x224/results/ensemble_late_fusion/validation_probability_screening/validation_model_screening_single_models.csv
Ranking ensemble: /content/drive/MyDrive/Dataset Fargetta_224x224/results/ensemble_late_fusion/validation_probability_screening/validation_probability_ensemble_screening_ranked.csv
Summary condensato: /content/drive/MyDrive/Dataset Fargetta_224x224/results/ensemble_late_fusion/validation_probability_screening/validation_probability_ensemble_screening_summary.csv
Selected for test: /content/drive/MyDrive/Dataset Fargetta_224x224/results/ensemble_late_fusion/validati

/tmp/ipykernel_566/1614859907.py:27: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(output_root)


Metadata compatibili v8 caricati/ricostruiti
Train: 14490 | Val: 816 | Final train: 14490 | Test: 816 | Classi: 46
Split info: {'mode': 'upsampling', 'train_source': '/content/drive/MyDrive/Dataset Fargetta_224x224/augmented_train_only_original_seed42_test0p1/train_upsampling_manifest.csv', 'val_source': 'reconstructed_from_seed', 'test_source': 'reconstructed_from_seed', 'final_train_source': '/content/drive/MyDrive/Dataset Fargetta_224x224/augmented_train_only_original_seed42_test0p1/train_upsampling_manifest.csv'}
Combinazioni candidate:
- resnet18__dinov2_small
- resnet18__dinov2_base
- dinov2_small__dinov2_base
- resnet18__dinov2_small__dinov2_base

Modello base: resnet18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Checkpoint caricato: resnet18_fold_fixed.pt | missing=0 | unexpected=0


...

Output dinov2_vit_base_patch14:   0%|          | 0/26 [00:00<?, ?it/s]

Salvati output dinov2_base | val: features=(816, 768), prob=(816, 46)
Checkpoint caricato: dinov2_vit_base_patch14_fold_fixed.pt | missing=0 | unexpected=0


Output dinov2_vit_base_patch14:   0%|          | 0/26 [00:00<?, ?it/s]

Salvati output dinov2_base | test: features=(816, 768), prob=(816, 46)

Late fusion embedding combo: resnet18__dinov2_small
Epoch 001 | loss=0.1040 | internal_val_acc=0.9167 | internal_val_f1=0.9081
Epoch 002 | loss=0.0032 | internal_val_acc=0.9191 | internal_val_f1=0.9084
Epoch 003 | loss=0.0026 | internal_val_acc=0.9154 | internal_val_f1=0.9056
Epoch 004 | loss=0.0015 | internal_val_acc=0.9167 | internal_val_f1=0.9080
Epoch 005 | loss=0.0013 | internal_val_acc=0.9191 | internal_val_f1=0.9099
Epoch 006 | loss=0.0026 | internal_val_acc=0.9179 | internal_val_f1=0.9099
Epoch 007 | loss=0.0026 | internal_val_acc=0.9179 | internal_val_f1=0.9091
Epoch 008 | loss=0.0036 | internal_val_acc=0.9191 | internal_val_f1=0.9081
Epoch 009 | loss=0.0016 | internal_val_acc=0.9216 | internal_val_f1=0.9108
Epoch 010 | loss=0.0014 | internal_val_acc=0.9142 | internal_val_f1=0.9037
Epoch 011 | loss=0.0012 | internal_val_acc=0.9154 | internal_val_f1=0.9054
Epoch 012 | loss=0.0009 | internal_val_acc=0.9179 |

,rank,ensemble_promising_label,ensemble_method,combo_size,combo_models,validation_screening_accuracy,validation_screening_f1_macro,validation_screening_balanced_accuracy,delta_validation_screening_accuracy_pp_vs_best_global_single,validation_screening_scope
0,1,very_promising,soft_voting_uniform,3,mobilenetv2|dinov2_vit_small_patch14|dinov2_vi...,0.949755,0.943295,0.938572,1.102941,holdout_validation
1,2,very_promising,weighted_soft_voting_per_class,3,mobilenetv2|dinov2_vit_small_patch14|dinov2_vi...,0.949755,0.943157,0.938572,1.102941,holdout_validation
2,3,very_promising,weighted_soft_voting_global,3,resnet18|dinov2_vit_small_patch14|dinov2_vit_b...,0.948529,0.945050,0.941031,0.980392,holdout_validation
3,4,very_promising,weighted_soft_voting_per_class,3,resnet18|dinov2_vit_small_patch14|dinov2_vit_b...,0.948529,0.944764,0.941031,0.980392,holdout_validation
4,5,very_promising,weighted_soft_voting_global,3,mobilenetv2|dinov2_vit_small_patch14|dinov2_vi...,0.948529,0.942853,0.937907,0.980392,holdout_validation
5,6,very_promising,weighted_soft_voting_global,3,vit_base_patch16_224|dinov2_vit_small_patch14|...,0.947304,0.944032,0.940164,0.857843,holdout_validation
6,7,very_promising,weighted_soft_voting_per_class,3,vit_base_patch16_224|dinov2_vit_small_patch14|...,0.947304,0.944032,0.940164,0.857843,holdout_validation
7,8,very_promising,weighted_soft_voting_per_class,2,resnet50|dinov2_vit_base_patch14,0.947304,0.943798,0.938958,0.857843,holdout_validation
